# Vroegsignalering — Effectiveness Analysis

Early-warning systems aim to reach households with debt before things escalate — but reaching
someone is only step one. This notebook asks *"did it work?"*: success means a household does
**not** come back with a new signal after its case is closed.

It uses three core tabs — **Signalen** (signals), **Meldingen** (cases), **Melders** (reporters) —
plus the contact-attempt tabs, linked by shared reference numbers.
**Privacy first:** every output is an aggregate statistic or chart; no individual records are shown.


## 0. Install Dependencies

Run once to install the required Python libraries. Skipped automatically if they are already present.


In [ ]:
%pip install pandas numpy matplotlib seaborn openpyxl statsmodels scipy --quiet


## 1. Setup

Imports all libraries, sets chart styling, and installs the report capture: everything printed,
displayed or plotted below is also collected into one HTML results file (written in §13).
**Update the `FILE` variable** in the cell below if the export has a different name or location.


In [ ]:
# Force a NON-blocking, headless plotting backend. Without this, plt.show() can
# block forever waiting for a GUI window (TkAgg/QtAgg/MacOSX) — which makes the
# first plotting cell appear to "hang". 'inline' renders in the notebook and is
# non-blocking; if IPython is unavailable we fall back to the headless 'Agg'.
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except Exception:
    import matplotlib
    matplotlib.use('Agg')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ── Configuration ─────────────────────────────────────────────────────────────
FILE     = Path('bi_export_sample.xlsx')   # ← update to real file path if needed
DATE_FMT = '%d-%m-%Y'

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
sns.set_theme(style='whitegrid')
COLORS = sns.color_palette('Set2')

# ══════════════════════════════════════════════════════════════════════════════
#  RESULTS-FILE CAPTURE
#  Every print(), every table shown with display(), and every chart shown with
#  plt.show() is automatically recorded AND written to a single self-contained
#  HTML file (elsa_results_report.html) at the end (Section 15).
#  Send that one file back — it opens in any browser and embeds all charts.
#  Cell outputs still appear in the notebook as normal.
# ══════════════════════════════════════════════════════════════════════════════
import io as _io, base64 as _b64, html as _html, datetime as _dt, builtins as _bi
from IPython.display import display as _ipy_display   # the REAL display (safe to re-import)

REPORT_PATH   = (FILE.parent if hasattr(FILE, 'parent') else Path('.')) / 'elsa_results_report.html'
_REPORT_BLOCKS = []

def _report_add(html_str):
    _REPORT_BLOCKS.append(html_str)

def _report_section(title):
    _report_add(f"<h2 class='sec'>{_html.escape(str(title))}</h2>")

# Tee print() → real stdout + report. builtins.print is always the genuine print,
# so re-running this cell never stacks wrappers on top of each other.
_orig_print = _bi.print
def print(*args, **kwargs):
    _orig_print(*args, **kwargs)
    sep = kwargs.get('sep', ' '); end = kwargs.get('end', '\n')
    _report_add(f"<pre>{_html.escape(sep.join(str(a) for a in args) + end)}</pre>")

# Tee display() → real display + report. _ipy_display is imported straight from
# IPython above, NOT taken from the namespace, so it can never become this wrapper
# (which previously caused infinite recursion when the cell was run twice).
def display(obj, *args, **kwargs):
    _ipy_display(obj, *args, **kwargs)
    try:
        if isinstance(obj, pd.DataFrame):
            _report_add(obj.to_html(border=0))
        elif isinstance(obj, pd.Series):
            _report_add(obj.to_frame().to_html(border=0))
        else:
            _report_add(f"<pre>{_html.escape(str(obj))}</pre>")
    except Exception:
        pass

# Tee plt.show() → embed the current figure as a PNG. Guarded so that re-running
# this cell does not wrap an already-wrapped show() (which would double-capture).
if not getattr(plt.show, '_elsa_wrapped', False):
    _orig_show = plt.show
    def _report_show(*args, **kwargs):
        fig = plt.gcf()
        try:
            buf = _io.BytesIO()
            fig.savefig(buf, format='png', dpi=110, bbox_inches='tight')
            buf.seek(0)
            _report_add(f"<img src='data:image/png;base64,{_b64.b64encode(buf.read()).decode()}'/>")
        except Exception:
            pass
        _orig_show(*args, **kwargs)
    _report_show._elsa_wrapped = True
    plt.show = _report_show


### 1a. Load Data

Reads every tab from the Excel file, parses date columns into a consistent format, and defines
the helper that converts Dutch `Ja`/`Nee` values to booleans. Row counts are printed in total and per year
(each tab on its own date column) — no data values.
Data-quality guard: placeholder dates (before 2000) are set to missing, with an
audit note printed into the report.


In [ ]:
_report_section('1a. Load Data')
xls = pd.ExcelFile(FILE)

signalen  = pd.read_excel(xls, 'Signalen')
meldingen = pd.read_excel(xls, 'Meldingen')
melders   = pd.read_excel(xls, 'Melders')

FIXED_TABS = {'Signalen', 'Meldingen', 'Melders'}
extra_tabs = {name: pd.read_excel(xls, name)
              for name in xls.sheet_names if name not in FIXED_TABS}

# ── Parse date columns (robust to already-datetime cells and other formats) ───
RAW_DATE_SAMPLES = {}   # original text of date columns, captured BEFORE parsing (for v2)
def parse_dates(df, cols):
    for c in cols:
        if c not in df.columns:
            continue
        s = df[c]
        if pd.api.types.is_datetime64_any_dtype(s):
            continue                      # Excel already gave real date cells
        ex = s.dropna().astype(str).head(3).tolist()   # example raw values reveal the format
        if ex:
            RAW_DATE_SAMPLES.setdefault(c, ex)
        parsed = pd.to_datetime(s, format=DATE_FMT, errors='coerce')
        if parsed.notna().mean() < 0.5:   # fixed format matched almost nothing
            parsed = pd.to_datetime(s, dayfirst=True, errors='coerce')
        df[c] = parsed

parse_dates(signalen,  ['Datum melding', 'Datum doorgezet', 'Startdatum contact'])
parse_dates(meldingen, ['Afgerond d.d.', 'Gezien d.d.'])
for df in extra_tabs.values():
    date_cols = [c for c in df.columns if 'datum' in c.lower() or c.lower() == 'datum']
    parse_dates(df, date_cols)

# ── Null out placeholder dates (real export contains e.g. 1000-01-01) ─────────
_MIN_VALID_DATE = pd.Timestamp('2000-01-01')
for _name, _df in [('Signalen', signalen), ('Meldingen', meldingen), *extra_tabs.items()]:
    for _c in _df.columns:
        if pd.api.types.is_datetime64_any_dtype(_df[_c]):
            _n = int((_df[_c] < _MIN_VALID_DATE).sum())
            if _n:
                print(f"  NOTE: {_name}.{_c}: {_n} placeholder date(s) before "
                      f"{_MIN_VALID_DATE.date()} set to missing")
                _df.loc[_df[_c] < _MIN_VALID_DATE, _c] = pd.NaT

# ── Ja/Nee → boolean helper ───────────────────────────────────────────────────
def ja_nee(series):
    return series.astype(str).str.strip().str.lower().map({'ja': True, 'nee': False})

# ── Confirm tabs loaded (row counts only — no raw data) ───────────────────────
all_tabs = [('Signalen', signalen), ('Meldingen', meldingen), ('Melders', melders),
            *extra_tabs.items()]
for name, df in all_tabs:
    print(f"  {name:<35} {len(df):>7,} rows")

# ── Rows per year, per tab (each counted on its own primary date column) ──────
_DATECOL = {'Signalen': 'Datum melding', 'Meldingen': 'Afgerond d.d.',
            'Tussenresultaat': 'Datum contactpoging', 'Eindresultaat': 'Datum contactpoging',
            'Follow-Up resultaat': 'Datum'}
_ycols = {}
for name, df in all_tabs:
    _dc = _DATECOL.get(name)
    if _dc and _dc in df.columns and pd.api.types.is_datetime64_any_dtype(df[_dc]):
        _vc = df[_dc].dt.year.value_counts()
        if len(_vc):
            _ycols[name] = _vc
if _ycols:
    _yt = pd.DataFrame(_ycols).fillna(0).astype(int).sort_index()
    _yt.index = _yt.index.astype(int)
    print("\n  Rows per year, per tab:")
    print(_yt.to_string())
    print("  (Signalen counted on signal date; Meldingen on close date; contact tabs on")
    print("   attempt date; Melders is a dateless registry. Rows with missing dates excluded.)")


## 2. Dataset Profile

The real export is seen only once, and only the HTML report comes back. This section therefore
records a privacy-safe profile of the dataset — schema, exact category labels, value ranges,
date coverage and join quality — so the next version of the analysis can be written without
needing the raw data again.


### 2a. Schema

For each tab: columns, data types, missing-value percentage, and number of distinct values.


In [ ]:
_report_section('2a. Schema — tabs, columns, dtypes, missingness, cardinality')
# Columns that must never have their raw values shown (identifiers / personal / free text)
SENSITIVE_COLS = {
    'Hash', 'SignaalID', 'Meldingnummer', 'ID', 'Melder ID', 'Postcode',
    'Naam medewerker', 'Melder', 'Melder (submerk)', 'Onderwerp',
    'Toelichting bij de FollowUp', 'Verwijzing anders namelijk', 'Waarnaar is verwezen',
}
CATEGORICAL_MAX = 50   # only list distinct values for columns with at most this many

def schema_table(df):
    return pd.DataFrame({
        'dtype':    df.dtypes.astype(str),
        'non_null': df.count(),
        'null_%':   (df.isnull().sum() / max(len(df), 1) * 100).round(1),
        'n_unique': df.nunique(dropna=True),
    })

for name, df in all_tabs:
    print(f"\n══ {name}: {len(df):,} rows × {df.shape[1]} cols ══")
    display(schema_table(df))


### 2b. Category Labels & Frequencies

The exact distinct values and their counts for every low-cardinality, non-identifier column —
pins down the precise spelling of contact methods, outcomes and Ja/Nee tokens for future code.


In [ ]:
_report_section('2b. Exact Category Labels & Frequencies')
for name, df in all_tabs:
    shown_any = False
    for col in df.columns:
        if col in SENSITIVE_COLS:
            continue
        nu = df[col].nunique(dropna=True)
        if nu < 1 or nu > 1000:
            continue
        if not shown_any:
            print(f"\n══ {name} ══")
            shown_any = True
        if nu <= CATEGORICAL_MAX:
            vc = df[col].value_counts(dropna=False)
            vc.index = vc.index.map(lambda x: '(blank)' if pd.isna(x) else str(x))
            print(f"\n• {col}  ({nu} distinct):")
            print(vc.to_string())
        else:
            # 50 < distinct <= 1000: list only the most common values
            vc = df[col].value_counts().head(15)
            print(f"\n• {col}  ({nu} distinct — showing top 15):")
            print(vc.to_string())


### 2c. Numeric Ranges & Date Coverage

Distribution of numeric columns (to design realistic bands) and, for each date column,
how well it parsed and what period it covers.


In [ ]:
_report_section('2c. Numeric Ranges & Date Coverage')
for name, df in all_tabs:
    num = df.select_dtypes(include='number')
    num = num[[c for c in num.columns if c not in SENSITIVE_COLS]]
    if num.shape[1]:
        print(f"\n══ {name} — numeric columns ══")
        display(num.describe().T[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']].round(2))

print("\n══ Date columns — coverage & parse success ══")
date_cols = {'Signalen': ['Datum melding', 'Datum doorgezet', 'Startdatum contact'],
             'Meldingen': ['Afgerond d.d.', 'Gezien d.d.']}
tab_lookup = dict(all_tabs)
for tab, cols in date_cols.items():
    if tab not in tab_lookup:
        continue
    df = tab_lookup[tab]
    for c in cols:
        if c in df.columns and pd.api.types.is_datetime64_any_dtype(df[c]):
            s = df[c]
            ok = s.notna().mean() * 100
            print(f"  {tab}.{c:<20} parsed={ok:5.1f}%  "
                  f"range={s.min().date() if s.notna().any() else 'n/a'} → "
                  f"{s.max().date() if s.notna().any() else 'n/a'}")

print("\n══ Raw date formats (original text, before parsing) ══")
if RAW_DATE_SAMPLES:
    for _col, _ex in RAW_DATE_SAMPLES.items():
        print(f"  {_col:<22} e.g. {_ex}")
else:
    print("  All date columns were already proper date cells (no text parsing needed).")

### 2d. Join Integrity & Structure

Checks how the tabs link together (orphan and duplicate rates) and the structures that decide
what is analysable: signals per household, contact attempts per case, and how many closed cases
have enough follow-up time.


In [ ]:
_report_section('2d. Join Integrity & Structural Distributions')
print("── Join integrity ──")
sig_with_mld = signalen['Meldingnummer'].notna().mean() * 100
print(f"  Signalen with a Meldingnummer (linkable to Meldingen): {sig_with_mld:.1f}%")

mld_keys = set(meldingen['Meldingnummer'].dropna())
for tab in ['Tussenresultaat', 'Eindresultaat', 'Follow-Up resultaat']:
    if tab in extra_tabs and 'Meldingnummer' in extra_tabs[tab].columns:
        ids = extra_tabs[tab]['Meldingnummer'].dropna()
        match = ids.isin(mld_keys).mean() * 100 if len(ids) else 0
        print(f"  {tab}: {len(ids):,} rows, {match:.1f}% match a Meldingen case")

if 'Melder ID' in signalen.columns:
    melder_ids = set(melders['ID'].dropna())
    m = signalen['Melder ID'].dropna()
    print(f"  Signalen.Melder ID matching Melders registry: "
          f"{m.isin(melder_ids).mean()*100:.1f}%")

dup_mld = int(meldingen['Meldingnummer'].duplicated().sum())
print(f"  Duplicate Meldingnummer in Meldingen: {dup_mld:,}")

print("\n── Recidive structure (signals per household Hash) ──")
sph = signalen.dropna(subset=['Hash']).groupby('Hash').size()
if len(sph):
    print(f"  Households (unique Hash): {len(sph):,}")
    print(f"  Signals per household: mean={sph.mean():.2f}  median={sph.median():.0f}  p99={sph.quantile(.99):.0f}")
    multi = (sph > 1).mean() * 100
    print(f"  Households with >1 signal (recidive present): {multi:.1f}%")

print("\n── Contact attempts per case ──")
att = []
for tab in ['Tussenresultaat', 'Eindresultaat']:
    if tab in extra_tabs:
        att.append(extra_tabs[tab][['Meldingnummer']])
if att:
    apc = pd.concat(att).dropna().groupby('Meldingnummer').size()
    print(f"  Cases with ≥1 attempt: {len(apc):,}")
    print(f"  Attempts per case: mean={apc.mean():.2f}  median={apc.median():.0f}  p99={apc.quantile(.99):.0f}")

print("\n── Follow-up availability for recidive windows ──")
if 'Afgerond d.d.' in meldingen.columns and pd.api.types.is_datetime64_any_dtype(meldingen['Afgerond d.d.']):
    dmax = signalen['Datum melding'].max()
    closed_ct = ja_nee(meldingen['Afgerond'])
    for label, days in {'6 months': 182, '9 months': 273}.items():
        ok = (closed_ct.fillna(False) &
              meldingen['Afgerond d.d.'].notna() &
              ((meldingen['Afgerond d.d.'] + pd.Timedelta(days=days)) <= dmax)).sum()
        print(f"  Closed cases with ≥{label} follow-up: {ok:,}")

print("\n── Cases per municipality (multilevel feasibility) ──")
if 'Gemeente' in meldingen.columns:
    cpm = meldingen['Gemeente'].value_counts()
    print(f"  Municipalities: {cpm.size}  |  cases/municipality "
          f"mean={cpm.mean():.0f} median={cpm.median():.0f} min={cpm.min()} max={cpm.max()}")

print("\n── One household per case? (Meldingnummer -> Hash) ──")
_sl  = signalen.dropna(subset=['Meldingnummer', 'Hash'])
_hpc = _sl.groupby('Meldingnummer')['Hash'].nunique()
_multi = int((_hpc > 1).sum())
print(f"  Cases with linked signals: {len(_hpc):,}")
print(f"  ...mapping to >1 distinct household Hash: {_multi:,} "
      f"({_multi / max(len(_hpc), 1) * 100:.2f}%)")
print("  OK - one household per case; case-level recidive logic holds." if _multi == 0
      else "  WARNING - some cases span multiple households; revisit the one-Hash-per-case assumption.")

print("\n── Signalen -> Meldingen referential integrity ──")
_sig_mld = signalen['Meldingnummer'].dropna()
_orphan  = int((~_sig_mld.isin(mld_keys)).sum())
print(f"  Signals carrying a Meldingnummer: {len(_sig_mld):,}")
print(f"  ...whose Meldingnummer is NOT found in Meldingen (orphans): {_orphan:,} "
      f"({_orphan / max(len(_sig_mld), 1) * 100:.2f}%)")

_j = signalen.dropna(subset=['Datum melding']).copy()
_j['_yr'] = _j['Datum melding'].dt.year
if _j['_yr'].nunique() > 1:
    _jt = _j.groupby('_yr').agg(signals=('Meldingnummer', 'size'),
                                linkable=('Meldingnummer', lambda s: s.notna().sum()))
    _jt['linkable_%'] = (_jt['linkable'] / _jt['signals'] * 100).round(1)
    print("\n── Linkable signals per year (data-quality evolution) ──")
    print(_jt.to_string())

### 2e. R0 — What does the system `Recidive` flag mean?

Verifies the semantics of the system-provided recurrence columns before anything relies on
them. **Step 1:** is `Recidive` exactly the union of `Recidive maand-1..6`? (agreement, kappa,
confusion matrix, mismatch-by-year for the 2021 boundary). **Step 2:** which rule generates the
flags — reconstructions from observed history for every candidate: {any creditor vs same
creditor} × {rolling window vs calendar month}, scored per flag. The best-matching rule is the
working definition; §7a/§11 and `recidive_hist` are read accordingly. `Schuiver`/`Match` are
noted but out of scope.


In [ ]:
_report_section('2e. R0 - What does the system Recidive flag mean?')
_mcols = [c for c in signalen.columns if c.startswith('Recidive maand')]
if 'Recidive' not in signalen.columns or not _mcols:
    print('Recidive columns not present in this export - R0 skipped.')
else:
    _sys = ja_nee(signalen['Recidive']).fillna(False)
    _union = pd.Series(False, index=signalen.index)
    for _c in _mcols:
        _union = _union | ja_nee(signalen[_c]).fillna(False)
    n = len(signalen)
    tp = int((_sys & _union).sum());  tn = int((~_sys & ~_union).sum())
    fp = int((~_sys & _union).sum()); fn = int((_sys & ~_union).sum())
    agree = (tp + tn) / n * 100
    _po = (tp + tn) / n
    _pe = (_sys.mean() * _union.mean()) + ((1 - _sys.mean()) * (1 - _union.mean()))
    kappa = (_po - _pe) / (1 - _pe) if _pe < 1 else float('nan')

    print(f"System Recidive = Ja:        {int(_sys.sum()):,} ({_sys.mean()*100:.1f}%)")
    print(f"Union(maand-1..{len(_mcols)}) = Ja:     {int(_union.sum()):,} ({_union.mean()*100:.1f}%)")
    print(f"\nExact agreement: {agree:.2f}%   Cohen's kappa: {kappa:.3f}   (n={n:,})")
    print("\nConfusion matrix (rows = system Recidive, cols = union of maand flags):")
    print(pd.DataFrame([[tn, fp], [fn, tp]],
                       index=['system Nee', 'system Ja'],
                       columns=['union Nee', 'union Ja']).to_string())

    if agree >= 99.5:
        print("\nCONCLUSION (step 1): Recidive is (near-)exactly the union of the maand flags ->")
        print("a 6-MONTH WINDOWED recurrence flag. Read 7a/'11 targeting as 'repeats within")
        print("~6 months'; full history is controlled via prior_band.")
    elif agree >= 95:
        print("\nCONCLUSION (step 1): Recidive largely matches the union of the maand flags;")
        print("the remainder points to edge rules (window boundaries / same-day ties).")
    else:
        print("\nCONCLUSION (step 1): Recidive does NOT match the union of the maand flags -")
        print("it follows another rule; step 2 below tries to identify it.")

    _yr = signalen['Datum melding'].dt.year
    _mism = (_sys != _union)
    if _mism.any() and _yr.notna().any():
        _tb = pd.DataFrame({'year': _yr, 'mismatch': _mism}).dropna()
        print("\nMismatch rate by signal year (union vs system):")
        print(_tb.groupby('year')['mismatch']
                 .agg(n='size', mismatch_pct=lambda s: round(s.mean()*100, 2)).to_string())

    # ── Step 2: WHICH rule generates the flags? Reconstruct candidates ─────────
    # Candidates per flag: {any-creditor | same-creditor} x {rolling window | calendar month}
    import bisect as _bi
    _sh = signalen.dropna(subset=['Hash', 'Datum melding']).copy()
    if 'Melder ID' in _sh.columns and 'Type melder' in melders.columns:
        _tymap = dict(zip(melders['ID'], melders['Type melder']))
        _sh['_ty'] = _sh['Melder ID'].map(_tymap)
    else:
        _sh['_ty'] = 'onbekend'
    _sh['_mo'] = _sh['Datum melding'].dt.year * 12 + _sh['Datum melding'].dt.month

    _shs = _sh.sort_values('Datum melding')
    _hist    = _shs.groupby('Hash')['Datum melding'].apply(list).to_dict()
    _hist_ty = _shs.groupby(['Hash', '_ty'])['Datum melding'].apply(list).to_dict()
    _mons    = _sh.groupby('Hash')['_mo'].agg(set).to_dict()
    _mons_ty = _sh.groupby(['Hash', '_ty'])['_mo'].agg(set).to_dict()

    def _roll(arr, t, lo_d, hi_d):
        if not arr or pd.isna(t): return False
        lo = t - pd.Timedelta(days=hi_d); hi = t - pd.Timedelta(days=lo_d)
        return _bi.bisect_right(arr, hi) - _bi.bisect_left(arr, lo) > 0
    def _cal(ms, mo, k1, k2):
        if not ms: return False
        return any((mo - k) in ms for k in range(k1, k2 + 1))

    _H  = list(zip(_sh['Hash'], _sh['_ty'], _sh['Datum melding'], _sh['_mo']))
    _cand = {}
    _cand[('Recidive', 'any/rolling 1-186d')]  = [_roll(_hist.get(h), t, 1, 186) for h, ty, t, mo in _H]
    _cand[('Recidive', 'any/calendar M-1..6')] = [_cal(_mons.get(h), mo, 1, 6) for h, ty, t, mo in _H]
    _cand[('Recidive', 'same-cred/rolling')]   = [_roll(_hist_ty.get((h, ty)), t, 1, 186) for h, ty, t, mo in _H]
    _cand[('Recidive', 'same-cred/calendar')]  = [_cal(_mons_ty.get((h, ty)), mo, 1, 6) for h, ty, t, mo in _H]
    for _k in (1, 2, 3):
        _lab = f'maand-{_k}'
        _cand[(_lab, f'any/rolling {(_k-1)*30+1}-{_k*30+7}d')] = \
            [_roll(_hist.get(h), t, (_k-1)*30+1, _k*30+7) for h, ty, t, mo in _H]
        _cand[(_lab, f'any/calendar M-{_k}')]  = [_cal(_mons.get(h), mo, _k, _k) for h, ty, t, mo in _H]
        _cand[(_lab, f'same-cred/calendar M-{_k}')] = [_cal(_mons_ty.get((h, ty)), mo, _k, _k) for h, ty, t, mo in _H]

    _sysflags = {'Recidive': _sys.loc[_sh.index]}
    for _k in (1, 2, 3):
        _cn = f'Recidive maand-{_k}'
        if _cn in signalen.columns:
            _sysflags[f'maand-{_k}'] = ja_nee(signalen[_cn]).fillna(False).loc[_sh.index]

    _obs0 = _sh['Datum melding'].min()
    _mask = (_sh['Datum melding'] >= _obs0 + pd.Timedelta(days=186)).values
    if int(_mask.sum()) >= 200:
        print(f"\nStep 2 - which rule generates each flag? (agreement %, signals from "
              f"{(_obs0 + pd.Timedelta(days=186)).date()} on; first 6 months excluded)")
        rows = []
        for (_flag, _rule), _vals in _cand.items():
            if _flag not in _sysflags: continue
            _sv = _sysflags[_flag].values[_mask]
            _rv = np.asarray(_vals)[_mask]
            rows.append((_flag, _rule, round((_sv == _rv).mean()*100, 1),
                         round(_sv.mean()*100, 1), round(_rv.mean()*100, 1)))
        _rt = pd.DataFrame(rows, columns=['system flag', 'candidate rule', 'agreement %',
                                          'system Ja %', 'reconstructed Ja %'])
        display(_rt)
        print("Best-matching rule per flag:")
        for _flag, g in _rt.groupby('system flag'):
            _b = g.sort_values('agreement %', ascending=False).iloc[0]
            print(f"  {_flag:<10} -> {_b['candidate rule']:<26} ({_b['agreement %']}%)")
        print("\nRead: >=97% for one rule = that IS the definition. All rules mediocre =")
        print("internal XXLLNC state (e.g. pre-export history) - confirm the best rule with them.")
    else:
        print("\nStep 2 skipped - too few signals after the 6-month burn-in window.")

    # footnote: sibling system flags, out of R0 scope
    for _fc in ['Schuiver', 'Match']:
        if _fc in signalen.columns:
            _fv = ja_nee(signalen[_fc]).fillna(False)
            print(f"Note: system flag '{_fc}' = Ja for {_fv.mean()*100:.1f}% - not analysed here (out of scope).")


### 2f. R1 — Recidive Fingerprint Census (what the intake screen shows and hides)

The six maand flags are the household-history "fingerprint" a caseworker sees at intake.
This census describes the incoming stream — how many signals arrive as new households,
freshly recurring, building, **saturated (all six Ja: the flag's ceiling)**, intermittent, or
**fading (about to reset to "newcomer")** — and quantifies the flag's two blind spots against
our own full observed history: **false newcomers** (flag = Nee but real history older than
6 months) and **saturation**. The bridge table gives the honest correction between the
system's ~6-month repeat rate and the true ever-repeat rate (relevant for §7a/§11).


In [ ]:
_report_section('2f. R1 - Recidive Fingerprint Census')
_mcols = [c for c in signalen.columns if c.startswith('Recidive maand')]
if len(_mcols) < 6 or 'Recidive' not in signalen.columns:
    print('Recidive maand columns not present - R1 skipped.')
else:
    _mcols = sorted(_mcols, key=lambda c: int(c.split('-')[-1]))
    _F = pd.DataFrame({c: (ja_nee(signalen[c]) == True) for c in _mcols})
    _depth = _F.sum(axis=1)
    _m1 = _F[_mcols[0]]
    _sysR = ja_nee(signalen['Recidive']) == True
    _yr = signalen['Datum melding'].dt.year

    print('Depth at arrival - number of maand-1..6 flags = Ja on the incoming signal:')
    _dd = _depth.value_counts().sort_index()
    print(pd.DataFrame({'signals': _dd, '%': (_dd / len(signalen) * 100).round(1)}).to_string())

    # ── fingerprint pattern (exhaustive & disjoint) ────────────────────────────
    _alive = pd.Series(True, index=signalen.index)
    _streak = pd.Series(0, index=signalen.index)
    for _c in _mcols:
        _alive = _alive & _F[_c]
        _streak = _streak + _alive.astype(int)
    _consec = _streak == _depth      # all Ja's form one unbroken run starting at maand-1

    pattern = pd.Series('', index=signalen.index, dtype=object)
    pattern[_depth == 0] = '0. no recent history (0/6)'
    pattern[(_depth >= 1) & (~_m1)] = '4. fading (gap since last signal)'
    pattern[(_depth == 1) & _m1] = '1. starting (maand-1 only)'
    pattern[_depth.between(2, 5) & _m1 & _consec] = '2. building (consecutive run)'
    pattern[_depth.between(2, 5) & _m1 & (~_consec)] = '3. intermittent (on-off rhythm)'
    pattern[_depth == 6] = '5. saturated (6/6 - ceiling)'

    print('\nFingerprint pattern of incoming signals:')
    _pc = pattern.value_counts().sort_index()
    print(pd.DataFrame({'signals': _pc, '%': (_pc / len(signalen) * 100).round(1)}).to_string())
    print("'fading' = will look like a NEWCOMER next month | 'saturated' = flag cannot")
    print("distinguish month-7 trouble from year-4 chronic | 'intermittent' often reflects")
    print('bi-monthly creditor reporting rhythm (see H2).')

    if _yr.nunique() > 1:
        _tmp = pd.DataFrame({'y': _yr, 'p': pattern}).dropna(subset=['y'])
        _pt = pd.crosstab(_tmp['y'].astype(int), _tmp['p'], normalize='index') * 100
        print('\nPattern mix per year (%):')
        print(_pt.round(1).to_string())

    # ── blind spots, measured against our own full observed history ───────────
    import bisect as _bi
    _GAP = 14   # same re-reporting convention as MIN_GAP_DAYS in section 9
    _sh = signalen.dropna(subset=['Hash', 'Datum melding'])
    _hist = _sh.sort_values('Datum melding').groupby('Hash')['Datum melding'].apply(list).to_dict()
    def _n_prior(h, t, cutoff_days):
        arr = _hist.get(h)
        if not arr or pd.isna(t):
            return 0
        return _bi.bisect_left(arr, t - pd.Timedelta(days=cutoff_days))
    _has_old = pd.Series([_n_prior(h, t, 186) > 0
                          for h, t in zip(_sh['Hash'], _sh['Datum melding'])], index=_sh.index)
    _has_any = pd.Series([_n_prior(h, t, _GAP) > 0
                          for h, t in zip(_sh['Hash'], _sh['Datum melding'])], index=_sh.index)
    _sysR2 = _sysR.loc[_sh.index]
    _sat2  = (_depth.loc[_sh.index] == 6)
    _obs0  = _sh['Datum melding'].min()
    _mask  = _sh['Datum melding'] >= _obs0 + pd.Timedelta(days=186)   # burn-in: history invisible before

    if int(_mask.sum()) >= 200:
        _mNee = _mask & (~_sysR2)
        _fn = (~_sysR2) & _has_old & _mask
        print('\nBlind spot 1 - FALSE NEWCOMERS (flag = Nee, but observed history older than 6 months):')
        print(f"  {int(_fn.sum()):,} of {int(_mNee.sum()):,} flag-Nee signals "
              f"({int(_fn.sum()) / max(int(_mNee.sum()), 1) * 100:.1f}%) - the 6-month window has")
        print('  reset on households that are in fact repeat households.')
        print('\nBlind spot 2 - SATURATION (all six flags Ja):')
        print(f"  {int((_sat2 & _mask).sum()):,} of {int(_mask.sum()):,} signals "
              f"({(_sat2 & _mask).sum() / max(int(_mask.sum()), 1) * 100:.1f}%) arrive at the flag's ceiling.")

        _bt = pd.DataFrame({'year': _yr.loc[_sh.index][_mask],
                            'flag Ja %': _sysR2[_mask].astype(float),
                            'observed ever-repeat %': _has_any[_mask].astype(float)}).dropna(subset=['year'])
        if _bt['year'].nunique() > 1:
            _g = (_bt.assign(year=_bt['year'].astype(int)).groupby('year').mean() * 100).round(1)
            _g['undercount (pp)'] = (_g['observed ever-repeat %'] - _g['flag Ja %']).round(1)
            print('\nBridge: system flag vs full-history repeat rate (after 6-month burn-in):')
            print(_g.to_string())
        else:
            print(f"\nBridge (after burn-in): flag Ja = {_sysR2[_mask].mean()*100:.1f}%  vs  "
                  f"observed ever-repeat = {_has_any[_mask].mean()*100:.1f}%")
        print('Use the observed column when the question is "was this household seen before";')
        print('the flag answers only "active in the last ~6 months" - read section 11 accordingly.')
    else:
        print('\nBlind-spot analysis skipped - too few signals after the 6-month burn-in window.')


## 3. Signal Volume & Distribution

How many signals arrive per month, and which signal types are most common.


In [ ]:
_report_section('3. Signal Volume & Distribution')
# Use only valid dates; convert to monthly Timestamps for a clean, fast x-axis.
# (Garbage/NaT dates are dropped so a few bad rows can't blow up the plot range.)
_valid_dates = signalen['Datum melding'].dropna()
monthly = (_valid_dates.dt.to_period('M').dt.to_timestamp()
           .value_counts().sort_index().rename('signals'))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if len(monthly) > 0:
    monthly.plot(ax=axes[0], marker='o', color=COLORS[0], linewidth=2)
axes[0].set_title('Signal Volume per Month')
axes[0].set_xlabel('')
axes[0].set_ylabel('Number of signals')

type_counts = signalen['Type melding'].value_counts()
type_counts.plot.barh(ax=axes[1], color=COLORS[1])
axes[1].set_title('Signal Type (Type melding)')
axes[1].set_xlabel('Count')

plt.tight_layout()
plt.show()

print(f"Total signals      : {len(signalen):,}")
print(f"Valid signal dates : {len(_valid_dates):,} "
      f"({len(_valid_dates)/max(len(signalen),1)*100:.1f}%)")
if len(_valid_dates) > 0:
    print(f"Date range         : {_valid_dates.min().date()} → {_valid_dates.max().date()}")
else:
    print("Date range         : no parseable dates in 'Datum melding'")
print()
print("Signal type counts:")
print(type_counts.to_string())
print()
print("Signals per month (values behind the monthly volume chart):")
print(monthly.to_string() if len(monthly) else "(no parseable dates)")
print()
print("Signals per year:")
print(_valid_dates.dt.year.value_counts().sort_index().rename('signals').to_string())


### 3a. Reporter Analysis

Which organisations (housing corporations, energy companies, health insurers…) send the signals,
and in what volume.


In [ ]:
_report_section('3a. Reporter (Melder) Analysis')
sig_with_melder = signalen.merge(
    melders.rename(columns={'ID': 'Melder ID'}), on='Melder ID', how='left'
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

top_reporters = sig_with_melder['Melder'].value_counts().head(15)
top_reporters.sort_values().plot.barh(ax=axes[0], color=COLORS[2])
axes[0].set_title('Top 15 Reporters by Signal Volume')
axes[0].set_xlabel('Number of signals')

type_melder = sig_with_melder['Type melder'].value_counts()
type_melder.plot.pie(ax=axes[1], autopct='%1.1f%%',
                     colors=sns.color_palette('Set2', len(type_melder)),
                     startangle=90)
axes[1].set_title('Reporter Type Distribution (Type melder)')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

print("Top 15 reporters by signal volume (values behind the left chart):")
print(top_reporters.sort_values(ascending=False).rename('signals').to_string())
print()
print("Reporter type — signal counts:")
print(
    type_melder.rename('signals')
    .to_frame()
    .assign(pct_of_total=(type_melder / len(signalen) * 100).round(1))
    .to_string()
)
_yr3 = sig_with_melder.dropna(subset=['Datum melding']).copy()
_yr3['_yr'] = _yr3['Datum melding'].dt.year
if _yr3['_yr'].nunique() > 1:
    _pv3 = (_yr3.pivot_table(index='Type melder', columns='_yr', values='Hash', aggfunc='count')
            .fillna(0).astype(int))
    print()
    print("Reporter type x year (signal counts):")
    print(_pv3.to_string())


## 4. Signal → Report Conversion

A signal is "forwarded" (`Doorgezet = Ja`) when the municipality decides it deserves follow-up.
This measures the overall forwarding rate and whether it differs by signal type or reporter type.


In [ ]:
_report_section('4. Signal → Report Conversion')
signalen['doorgezet_bool'] = ja_nee(signalen['Doorgezet'])
signalen['has_report']     = signalen['Meldingnummer'].notna()

total       = len(signalen)
n_forwarded = int(signalen['doorgezet_bool'].sum())

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Overall pie
doorgezet_counts = signalen['Doorgezet'].value_counts(dropna=False)
doorgezet_counts.index = doorgezet_counts.index.fillna('(blank)')
doorgezet_counts.plot.pie(ax=axes[0], autopct='%1.1f%%',
                          colors=[COLORS[1], COLORS[0], COLORS[3]], startangle=90)
axes[0].set_title('Forwarded to Report?\n(Doorgezet)')
axes[0].set_ylabel('')

# By signal type
type_conv = (signalen.groupby('Type melding')['doorgezet_bool']
             .agg(forwarded='sum', total='count'))
type_conv['rate_%'] = type_conv['forwarded'] / type_conv['total'] * 100
type_conv['rate_%'].sort_values().plot.barh(ax=axes[1], color=COLORS[3])
axes[1].axvline(n_forwarded / total * 100, color='red', linestyle='--',
                linewidth=1.5, label=f'Overall {n_forwarded/total*100:.1f}%')
axes[1].xaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].set_title('Conversion Rate by Signal Type')
axes[1].set_xlabel('% Forwarded')
axes[1].legend(fontsize=9)

# By reporter type
if 'Type melder' in sig_with_melder.columns:
    sig_with_melder['doorgezet_bool'] = ja_nee(sig_with_melder['Doorgezet'])
    type_melder_conv = (sig_with_melder.groupby('Type melder')['doorgezet_bool']
                        .agg(forwarded='sum', total='count'))
    type_melder_conv['rate_%'] = (type_melder_conv['forwarded']
                                  / type_melder_conv['total'] * 100)
    type_melder_conv['rate_%'].sort_values().plot.barh(ax=axes[2], color=COLORS[4])
    axes[2].axvline(n_forwarded / total * 100, color='red', linestyle='--', linewidth=1.5)
    axes[2].xaxis.set_major_formatter(mtick.PercentFormatter())
    axes[2].set_title('Conversion Rate by Reporter Type')
    axes[2].set_xlabel('% Forwarded')

plt.tight_layout()
plt.show()

print(f"Overall forwarding rate: {n_forwarded:,} / {total:,} = {n_forwarded/total*100:.1f}%")
print()
print("Conversion by signal type:")
print(type_conv.sort_values('rate_%', ascending=False)
      .assign(**{'rate_%': type_conv['rate_%'].round(1)})
      .to_string())
print()
print("Forwarded vs not (Doorgezet) - counts behind the pie:")
print(doorgezet_counts.rename('count').to_string())
if 'Type melder' in sig_with_melder.columns:
    print()
    print("Conversion by reporter type (values behind the right chart):")
    print(type_melder_conv.sort_values('rate_%', ascending=False)
          .assign(**{'rate_%': type_melder_conv['rate_%'].round(1)}).to_string())

_y4 = signalen.dropna(subset=['Datum melding']).copy()
_y4['_yr'] = _y4['Datum melding'].dt.year
_y4['_dz'] = ja_nee(_y4['Doorgezet']) == True
if _y4['_yr'].nunique() > 1:
    _yt4 = _y4.groupby('_yr')['_dz'].agg(total='count', forwarded='sum')
    _yt4['rate_%'] = (_yt4['forwarded'] / _yt4['total'] * 100).round(1)
    print()
    print("Forwarding rate per year:")
    print(_yt4.to_string())

    if _y4['Type melding'].notna().any():
        _p4t = _y4.pivot_table(index='Type melding', columns='_yr', values='_dz', aggfunc='mean') * 100
        _n4t = _y4.pivot_table(index='Type melding', columns='_yr', values='_dz', aggfunc='count')
        print("\nForwarding rate by signal type per year (%, cells n>=30):")
        print(_p4t.where(_n4t >= 30).round(1).to_string())
    _ym4 = sig_with_melder.dropna(subset=['Datum melding']).copy()
    _ym4['_yr'] = _ym4['Datum melding'].dt.year
    _ym4['_dz'] = ja_nee(_ym4['Doorgezet']) == True
    if 'Type melder' in _ym4.columns and _ym4['Type melder'].notna().any():
        _p4m = _ym4.pivot_table(index='Type melder', columns='_yr', values='_dz', aggfunc='mean') * 100
        _n4m = _ym4.pivot_table(index='Type melder', columns='_yr', values='_dz', aggfunc='count')
        print("\nForwarding rate by reporter type per year (%, cells n>=30):")
        print(_p4m.where(_n4m >= 30).round(1).to_string())


## 5. Contact Effectiveness

Once a case exists, the municipality tries to reach the client. This section measures reach —
overall, per contact method, and per time of day — using `Persoon bereikt` from the contact
tabs. (The `Bereikt` field in Meldingen is ~100% "N.v.t." for this municipality, so it is not used.)


In [ ]:
_report_section('5. Contact Effectiveness')
# Reach is measured from contact attempts (Persoon bereikt); Meldingen 'Bereikt' is
# ~100% 'N.v.t.' for this municipality and is therefore not used.
contact_dfs = []
for tab_name in ['Tussenresultaat', 'Eindresultaat']:
    if tab_name in extra_tabs:
        df = extra_tabs[tab_name].copy()
        df['_tab'] = tab_name
        contact_dfs.append(df)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

if contact_dfs:
    contacts = pd.concat(contact_dfs, ignore_index=True)
    contacts['bereikt_bool'] = ja_nee(contacts['Persoon bereikt'])

    # Case-level reach: a case counts as reached if ANY attempt reached the person
    case_reach = contacts.groupby('Meldingnummer')['bereikt_bool'].any()
    meldingen['reached_case'] = meldingen['Meldingnummer'].map(case_reach).fillna(False)
    attempt_rate = contacts['bereikt_bool'].mean() * 100

    reached_ct = pd.Series({'Reached': int(case_reach.sum()),
                            'Not reached': int((~case_reach).sum())})
    reached_ct.plot.pie(ax=axes[0], autopct='%1.1f%%',
                        colors=[COLORS[0], COLORS[1]], startangle=90)
    axes[0].set_title('Client Reached? (case-level)\nany successful attempt'); axes[0].set_ylabel('')

    method_reach = (contacts.groupby('Wijze van contactpoging')['bereikt_bool']
                    .agg(reach_rate='mean', n_attempts='count'))
    method_reach['reach_rate'] *= 100
    method_reach['reach_rate'].sort_values().plot.barh(ax=axes[1], color=COLORS[2])
    axes[1].axvline(attempt_rate, color='red', linestyle='--', linewidth=1.5, label='Overall avg')
    axes[1].xaxis.set_major_formatter(mtick.PercentFormatter())
    axes[1].set_title('Reach Rate by Contact Method'); axes[1].set_xlabel('% Person Reached'); axes[1].legend(fontsize=9)

    dagdeel_reach = (contacts.groupby('Dagdeel')['bereikt_bool']
                     .agg(reach_rate='mean', n_attempts='count'))
    dagdeel_reach['reach_rate'] *= 100
    dagdeel_reach['reach_rate'].sort_values().plot.barh(ax=axes[2], color=COLORS[4])
    axes[2].axvline(attempt_rate, color='red', linestyle='--', linewidth=1.5)
    axes[2].xaxis.set_major_formatter(mtick.PercentFormatter())
    axes[2].set_title('Reach Rate by Time of Day (Dagdeel)'); axes[2].set_xlabel('% Person Reached')

    plt.tight_layout(); plt.show()

    print("Reach is measured from contact attempts (Persoon bereikt); Meldingen 'Bereikt' is unused (~100% 'N.v.t.').")
    print(f"Case-level reach rate    : {case_reach.mean()*100:.1f}%  "
          f"({int(case_reach.sum()):,} of {case_reach.size:,} cases with >=1 attempt)")
    print(f"Attempt-level reach rate : {attempt_rate:.1f}%  ({len(contacts):,} attempts)")
    print()
    print("Reach rate by contact method:")
    print(method_reach.sort_values('reach_rate', ascending=False)
          .assign(reach_rate=method_reach['reach_rate'].round(1)).to_string())
    print()
    print("Reach rate by time of day (Dagdeel) - values behind the right chart:")
    print(dagdeel_reach.sort_values('reach_rate', ascending=False)
          .assign(reach_rate=dagdeel_reach['reach_rate'].round(1)).to_string())

    _cy = contacts.dropna(subset=['Datum contactpoging']).copy()
    _cy['_yr'] = pd.to_datetime(_cy['Datum contactpoging'], errors='coerce').dt.year
    _cy['_r']  = _cy['bereikt_bool'] == True
    _cy = _cy.dropna(subset=['_yr'])
    if _cy['_yr'].nunique() > 1:
        _yt5 = _cy.groupby('_yr')['_r'].agg(attempts='count', reach='mean')
        _yt5['reach_%'] = (_yt5['reach'] * 100).round(1)
        print("\nAttempt-level reach rate per year:")
        print(_yt5.drop(columns='reach').to_string())

        _cm5 = _cy.dropna(subset=['Wijze van contactpoging'])
        _p5m = _cm5.pivot_table(index='Wijze van contactpoging', columns='_yr', values='_r', aggfunc='mean') * 100
        _n5m = _cm5.pivot_table(index='Wijze van contactpoging', columns='_yr', values='_r', aggfunc='count')
        _p5m = _p5m.where(_n5m >= 100).dropna(how='all')
        if len(_p5m):
            print("\nReach rate by contact method per year (%, cells with n>=100 attempts):")
            print(_p5m.round(1).to_string())
        _cd5 = _cy.dropna(subset=['Dagdeel'])
        if len(_cd5) >= 200:
            _p5d = _cd5.pivot_table(index='Dagdeel', columns='_yr', values='_r', aggfunc='mean') * 100
            _n5d = _cd5.pivot_table(index='Dagdeel', columns='_yr', values='_r', aggfunc='count')
            _p5d = _p5d.where(_n5d >= 100).dropna(how='all')
            if len(_p5d):
                print("\nReach rate by time of day per year (%, cells with n>=100 attempts):")
                print(_p5d.round(1).to_string())
else:
    meldingen['reached_case'] = False
    for ax in axes:
        ax.text(0.5, 0.5, 'No contact-attempt tabs\nfound in this export',
                ha='center', va='center', transform=ax.transAxes, fontsize=11)
    plt.tight_layout(); plt.show()
    print("No contact-attempt tabs found in this export.")


## 6. Client Response & Outcomes

Among clients who were reached: do they want help (`Wil de klant hulp`),
and what is the final result of the case (`Wat is het eindresultaat`)?


In [ ]:
_report_section('6. Client Response & Outcomes')
if 'Eindresultaat' in extra_tabs:
    eind = extra_tabs['Eindresultaat'].copy()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    hulp_vc = eind['Wil de klant hulp'].value_counts(dropna=False)
    hulp_vc.index = hulp_vc.index.fillna('(not recorded)')
    hulp_vc.plot.pie(ax=axes[0], autopct='%1.1f%%',
                     colors=sns.color_palette('Set2', len(hulp_vc)), startangle=90)
    axes[0].set_title('Does the Client Want Help?\n(Wil de klant hulp)')
    axes[0].set_ylabel('')

    result_vc = eind['Wat is het eindresultaat'].value_counts(dropna=False)
    result_vc.index = result_vc.index.fillna('(not recorded)')
    result_vc.sort_values().plot.barh(ax=axes[1], color=COLORS[0])
    axes[1].set_title('Final Outcome (Wat is het eindresultaat)')
    axes[1].set_xlabel('Count')

    plt.tight_layout()
    plt.show()

    print("Does the client want help? (Wil de klant hulp) - counts behind the pie:")
    print(hulp_vc.rename('count').to_frame()
          .assign(pct_=(hulp_vc / hulp_vc.sum() * 100).round(1)).to_string())
    print()
    print("Outcome breakdown:")
    print(
        result_vc.rename('count')
        .to_frame()
        .assign(pct_=(result_vc / result_vc.sum() * 100).round(1))
        .sort_values('count', ascending=False)
        .to_string()
    )
    _ey = eind.dropna(subset=['Datum contactpoging']).copy()
    _ey['_yr'] = _ey['Datum contactpoging'].dt.year
    _ey['_no'] = _ey['Wat is het eindresultaat'].astype(str).str.contains('geen contact kunnen',
                                                                           case=False, na=False)
    if _ey['_yr'].nunique() > 1:
        _yt6 = _ey.groupby('_yr')['_no'].agg(cases='count', no_contact='mean')
        _yt6['no_contact_%'] = (_yt6['no_contact'] * 100).round(1)
        print("\n'Definitief geen contact kunnen krijgen' share per year:")
        print(_yt6.drop(columns='no_contact').to_string())

        _er6 = _ey['Wat is het eindresultaat'].fillna('(not recorded)').astype(str).str[:45]
        _p6r = pd.crosstab(_er6, _ey['_yr'], normalize='columns') * 100
        print("\nFinal result mix per year (% of that year's Eindresultaat rows):")
        print(_p6r.round(1).to_string())
        _wh6 = _ey['Wil de klant hulp'].fillna('(not recorded)').astype(str).str[:35]
        _p6w = pd.crosstab(_wh6, _ey['_yr'], normalize='columns') * 100
        print("\n'Wil de klant hulp' mix per year (%):")
        print(_p6w.round(1).to_string())
else:
    print("No 'Eindresultaat' tab found in this export.")


## 7. Case Resolution & Timing

Status overview of all cases (resolved, reached, withdrawn, incorrect) and how long it takes
from the first signal to case closure.


In [ ]:
_report_section('7. Case Resolution & Timing')
meldingen['afgerond_bool']    = ja_nee(meldingen['Afgerond'])
meldingen['ingetrokken_bool'] = ja_nee(meldingen['Ingetrokken'])
meldingen['onterecht_bool']   = ja_nee(meldingen['Onterecht'])
if 'reached_case' not in meldingen.columns:
    meldingen['reached_case'] = False

earliest_signal = (signalen.dropna(subset=['Meldingnummer'])
                   .groupby('Meldingnummer')['Datum melding'].min()
                   .rename('Datum melding'))
mel = meldingen.join(earliest_signal, on='Meldingnummer')
mel['days_to_close'] = (mel['Afgerond d.d.'] - mel['Datum melding']).dt.days

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

n_mel = len(meldingen)
status = pd.Series({
    'Resolved\n(Afgerond)':     int(meldingen['afgerond_bool'].sum()),
    'Reached\n(via contact)':   int(meldingen['reached_case'].sum()),
    'Withdrawn\n(Ingetrokken)': int(meldingen['ingetrokken_bool'].sum()),
    'Incorrect\n(Onterecht)':   int(meldingen['onterecht_bool'].sum()),
})
bars = axes[0].bar(status.index, status.values, color=COLORS[:4])
for bar in bars:
    h = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width() / 2, h + 0.3,
                 f'{int(h):,}\n({h/max(n_mel,1)*100:.1f}%)',
                 ha='center', va='bottom', fontsize=9)
axes[0].set_title(f'Report Status Overview  (n={n_mel:,})')
axes[0].set_ylabel('Count')

dtc = mel['days_to_close'].dropna()
dtc = dtc[dtc >= 0]
if len(dtc) > 1:
    dtc.hist(ax=axes[1], bins=min(30, len(dtc)), color=COLORS[2], edgecolor='white')
    axes[1].axvline(dtc.median(), color='red', linestyle='--', linewidth=1.5,
                    label=f'Median: {dtc.median():.0f}d')
    axes[1].axvline(dtc.mean(), color='orange', linestyle=':', linewidth=1.5,
                    label=f'Mean: {dtc.mean():.0f}d')
    axes[1].set_title('Days from Signal to Case Closure')
    axes[1].set_xlabel('Days'); axes[1].set_ylabel('Count'); axes[1].legend()
else:
    axes[1].text(0.5, 0.5, 'Insufficient data\nfor time-to-close',
                ha='center', va='center', transform=axes[1].transAxes, fontsize=11)

plt.tight_layout(); plt.show()

print("Status summary (of all cases in the export):")
for label, count in status.items():
    print(f"  {label.replace(chr(10), ' '):<30} {count:>6,}  ({count/max(n_mel,1)*100:.1f}%)")
print("  Note: 'Reached' is case-level from contact attempts; days-to-close covers only")
print("        cases with a linked signal in the observed window.")
if len(dtc) > 1:
    print(f"\nTime to close (days) - percentiles behind the histogram:")
    print(f"  mean={dtc.mean():.0f}  p10={dtc.quantile(.1):.0f}  p25={dtc.quantile(.25):.0f}  "
          f"p50={dtc.median():.0f}  p75={dtc.quantile(.75):.0f}  p90={dtc.quantile(.9):.0f}")

_myr = mel.dropna(subset=['days_to_close', 'Afgerond d.d.']).copy()
_myr = _myr[_myr['days_to_close'] >= 0]
_myr['_yr'] = _myr['Afgerond d.d.'].dt.year
if _myr['_yr'].nunique() > 1:
    _yt7a = _myr.groupby('_yr')['days_to_close'].agg(cases='count', median_days='median').round(0)
    print("\nTime to close per close-year:")
    print(_yt7a.to_string())

_ms7 = meldingen.dropna(subset=['Afgerond d.d.']).copy()
_ms7['_yr'] = _ms7['Afgerond d.d.'].dt.year
if _ms7['_yr'].nunique() > 1:
    _st7 = pd.DataFrame({
        'cases': _ms7.groupby('_yr').size(),
        'resolved %': _ms7.groupby('_yr')['afgerond_bool'].apply(lambda s: (s == True).mean() * 100),
        'reached %': _ms7.groupby('_yr')['reached_case'].apply(lambda s: (s == True).mean() * 100),
        'withdrawn %': _ms7.groupby('_yr')['ingetrokken_bool'].apply(lambda s: (s == True).mean() * 100),
    })
    print("\nCase status per close-year:")
    print(_st7.round(1).to_string())


### 7a. Recidive — Repeated Signals

A recidive signal means the same household signalled before. Shows what share of signals are
recidive (per month-prior flag) and whether repeat households are forwarded more or less often.


In [ ]:
_report_section('7a. Recidive — Repeated Signals')
recidive_cols = [c for c in signalen.columns if c.startswith('Recidive maand')]

if recidive_cols:
    rec_rates = {col.replace('Recidive ', ''): ja_nee(signalen[col]).mean() * 100
                 for col in recidive_cols}
    overall_rec = ja_nee(signalen['Recidive']).mean() * 100

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    pd.Series(rec_rates).plot.bar(ax=axes[0], color=COLORS[0])
    axes[0].axhline(overall_rec, color='red', linestyle='--', linewidth=1.5,
                    label=f'Overall: {overall_rec:.1f}%')
    axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())
    axes[0].set_title('Recidive Rate by Month Prior')
    axes[0].set_ylabel('% of signals flagged as recidive')
    axes[0].tick_params(axis='x', rotation=0)
    axes[0].legend()

    rec_conv = (signalen
                .groupby(ja_nee(signalen['Recidive']).rename('recidive'))
                ['doorgezet_bool'].mean() * 100)
    rec_conv.index = rec_conv.index.map({True: 'Recidive', False: 'No Recidive'})
    rec_conv.plot.bar(ax=axes[1], color=[COLORS[1], COLORS[2]])
    axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
    axes[1].set_title('Forwarding Rate: Recidive vs No Recidive')
    axes[1].set_ylabel('% Forwarded to Report (Doorgezet)')
    axes[1].tick_params(axis='x', rotation=0)

    plt.tight_layout()
    plt.show()

    print(f"Overall recidive rate: {overall_rec:.1f}%")
    print(pd.Series(rec_rates).round(1).rename('recidive_%').to_string())
    print()
    print("Forwarding rate by recidive status (values behind the right chart):")
    print(rec_conv.round(1).rename('forwarded_%').to_frame().to_string())

    _ry = signalen.dropna(subset=['Datum melding']).copy()
    _ry['_yr'] = _ry['Datum melding'].dt.year
    _ry['_rec'] = ja_nee(_ry['Recidive']) == True
    if _ry['_yr'].nunique() > 1:
        _yt7 = (_ry.groupby('_yr')['_rec'].mean() * 100).round(1)
        print("\nRecidive rate per signal year (%):")
        print(_yt7.rename('recidive_%').to_string())

        _pm7 = {}
        for _mc in recidive_cols:
            _ry['_mb'] = ja_nee(_ry[_mc]) == True
            _pm7[_mc.replace('Recidive ', '')] = _ry.groupby('_yr')['_mb'].mean() * 100
        print("\nMaand-flag rates per year (%):")
        print(pd.DataFrame(_pm7).round(1).to_string())
        _ry['_dz7'] = ja_nee(_ry['Doorgezet']) == True
        _pf7 = _ry.pivot_table(index='_rec', columns='_yr', values='_dz7', aggfunc='mean') * 100
        _pf7.index = _pf7.index.map({True: 'Recidive', False: 'No recidive'})
        print("\nForwarding rate by recidive status per year (%):")
        print(_pf7.round(1).to_string())
else:
    print("No recidive month columns found.")


## 8. Effectiveness Funnel

The full journey for one consistent cohort: signals received → forwarded → case exists →
client reached → wants help → case resolved. Restricted to cases originating from the observed
signal window so every stage shares one denominator (cases can outlive their signals, so a raw
funnel could exceed 100%).


In [ ]:
_report_section('8. End-to-End Effectiveness Funnel')
# Restrict to the CURRENT 6-month signal cohort so every stage shares one denominator.
# (Meldingen persist for years but old signals are deleted, so len(meldingen) is NOT a
#  subset of current signals; a raw signal->case funnel would exceed 100%.)
_dz = ja_nee(signalen['Doorgezet']) == True
cohort_mnums = set(signalen.loc[_dz, 'Meldingnummer'].dropna())
cohort_mld = meldingen[meldingen['Meldingnummer'].isin(cohort_mnums)]
if 'reached_case' not in cohort_mld.columns:
    cohort_mld = cohort_mld.assign(reached_case=False)

n_signals   = len(signalen)
n_forwarded = int(_dz.sum())
n_reports   = len(cohort_mld)
n_reached   = int(cohort_mld['reached_case'].sum())
n_resolved  = int(ja_nee(cohort_mld['Afgerond']).sum())

stages = ['Signals received (observed period)',
          'Forwarded to report\n(Doorgezet = Ja)',
          'Distinct case exists\n(signals can share a case)',
          'Client reached\n(via contact attempts)']
values = [n_signals, n_forwarded, n_reports, n_reached]

if 'Eindresultaat' in extra_tabs:
    e_co = extra_tabs['Eindresultaat']
    e_co = e_co[e_co['Meldingnummer'].isin(cohort_mnums)]
    hulp_ja = int(e_co['Wil de klant hulp'].fillna('').astype(str).str.lower().str.startswith('ja').sum())
    stages.append('Client wants help\n(Wil hulp = Ja)')
    values.append(hulp_ja)

stages.append('Case resolved\n(Afgerond = Ja)')
values.append(n_resolved)

palette = sns.color_palette('Blues_r', len(stages))
fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(range(len(stages)), values, color=palette)
for bar, val in zip(bars, values):
    pct = val / max(n_signals, 1) * 100
    ax.text(bar.get_width() + n_signals * 0.01, bar.get_y() + bar.get_height() / 2,
            f'{val:,}   ({pct:.1f}% of signals)', va='center', fontsize=10)
ax.set_yticks(range(len(stages))); ax.set_yticklabels(stages, fontsize=11)
ax.set_xlabel('Count')
ax.set_title('End-to-End Funnel - observed signal cohort', fontsize=14, pad=15)
ax.set_xlim(0, n_signals * 1.45); ax.invert_yaxis()
plt.tight_layout(); plt.show()

print("Funnel (signal cohort of the observed period; one consistent denominator):")
for stage, val in zip(stages, values):
    print(f"  {stage.replace(chr(10), ' '):<44} {val:>6,}  ({val/max(n_signals,1)*100:.1f}%)")
print("\nNote: cases whose signals predate the observed signal window are excluded -")
print("cases can outlive their signals, so a raw signal->case funnel could exceed 100%.")
print("Stage 3 counts distinct cases: several signals can share one case, so it is")
print("naturally smaller than stage 2 (a unit switch, not drop-out).")

_fy = signalen.dropna(subset=['Datum melding']).copy()
_fy['_yr'] = _fy['Datum melding'].dt.year
if _fy['_yr'].nunique() > 1:
    _rows_y = []
    for _yy, _gy in _fy.groupby('_yr'):
        _dzy = ja_nee(_gy['Doorgezet']) == True
        _mns = set(_gy.loc[_dzy, 'Meldingnummer'].dropna())
        _cm = meldingen[meldingen['Meldingnummer'].isin(_mns)]
        _rows_y.append({'year': int(_yy), 'signals': len(_gy),
                        'forwarded %': round(_dzy.mean() * 100, 1), 'cases': len(_cm),
                        'reached % of cases': round(_cm['reached_case'].mean() * 100, 1)
                            if len(_cm) and 'reached_case' in _cm.columns else np.nan,
                        'resolved % of cases': round((ja_nee(_cm['Afgerond']) == True).mean() * 100, 1)
                            if len(_cm) else np.nan})
    print("\nFunnel per signal year:")
    print(pd.DataFrame(_rows_y).set_index('year').to_string())


## 9. Effectiveness Framework — Does Contact Reduce Return?

Sections 9–12 measure **effectiveness**: not *"did we reach them?"* (Divosa already models that)
but *"did they stay away afterwards?"*. The outcome is the number of days from **case close**
until the same household (`Hash`) sends a **new signal**. Households with no new signal remain
in the analysis as signal-free for every observed day through the data cutoff. Their eventual
time-to-return is administratively right-censored at that cutoff because later events are not
yet observable; they are **not** classified as returned, unsuccessful, or lost to follow-up.
A household with a complete two-year window and no new signal therefore contributes as a
non-returner through two years, whereas a recently closed case contributes only its available
follow-up. With the export spanning 2021–2026, return
probabilities are reported up to **two years** after closure, and each case is adjusted for the
household's **observed prior history**. A new signal within 14 days of closure counts as
re-reporting of the same episode, not a return. Results are associational, not causal.

Absence of a signal is the desired observed outcome and may indicate resolution. It is not,
however, proof that the underlying financial problem was solved: moving away, identifier
changes, debts outside participating creditors, or departure from the reporting system can
look the same in these data. Section 13d examines this as *silent success vs disappearance*.

This cell builds the shared analysis frame: one row per closed, linked case, with its arm
(reached / refused / accepted), covariates, and time-to-return.


In [ ]:
_report_section('9. Effectiveness Framework — data preparation')
import numpy as np
import bisect
from statsmodels.duration.survfunc import SurvfuncRight, survdiff
from statsmodels.duration.hazard_regression import PHReg

EXPORT_DATE = signalen['Datum melding'].max()
print(f"Export (last signal) date: {EXPORT_DATE.date() if pd.notna(EXPORT_DATE) else 'n/a'}")

# A new signal arriving within this many days of case close is treated as re-reporting
# of the same episode (creditors deliver signals in monthly batches), not a genuine return.
MIN_GAP_DAYS = 14
print(f"Minimum gap for a 'return': >{MIN_GAP_DAYS} days after case close")

# ── Clean age (raw column contains impossible values e.g. -7973, 125) ─────────
signalen['Leeftijd_clean'] = signalen['Leeftijd'].where(signalen['Leeftijd'].between(18, 99))

# ── Signal-level helper flags ─────────────────────────────────────────────────
signalen['doorgezet_bool'] = ja_nee(signalen['Doorgezet'])
signalen['recidive_hist']  = ja_nee(signalen['Recidive'])   # built-in backward-looking flag

# reporter type onto signals
if 'Melder ID' in signalen.columns and 'Type melder' in melders.columns:
    _mel = melders.rename(columns={'ID': 'Melder ID'})[['Melder ID', 'Type melder']]
    signalen_x = signalen.merge(_mel, on='Melder ID', how='left')
else:
    signalen_x = signalen.copy(); signalen_x['Type melder'] = np.nan

# debt & age bands (ascii labels — safe for regression formulas)
_debt_col = next((c for c in ['Achterstandsbedrag', 'Termijnbedrag'] if c in signalen_x.columns), None)
if _debt_col:
    signalen_x['debt_band'] = pd.cut(signalen_x[_debt_col].clip(lower=0),
        bins=[-1, 200, 700, 2000, 1e12], labels=['<200', '200-700', '700-2k', '>2k'])
signalen_x['age_band'] = pd.cut(signalen_x['Leeftijd_clean'],
    bins=[18, 30, 45, 60, 100], right=False, labels=['18-29', '30-44', '45-59', '60+'])

# ── Case -> originating (earliest) signal + its covariates ────────────────────
_slink = signalen_x.dropna(subset=['Meldingnummer']).sort_values('Datum melding')
_first = (_slink.groupby('Meldingnummer', as_index=False).first()
          .rename(columns={'Datum melding': 'signal_date'}))

# ── Contact tabs -> case-level reach / attempts / first-reach date ─────────────
_cparts = []
for _t in ['Tussenresultaat', 'Eindresultaat']:
    if _t in extra_tabs:
        _d = extra_tabs[_t].copy(); _d['_tab'] = _t
        _cparts.append(_d)
contacts = pd.concat(_cparts, ignore_index=True) if _cparts else pd.DataFrame()

if len(contacts):
    contacts['_reached'] = ja_nee(contacts['Persoon bereikt'])
    _g = contacts.groupby('Meldingnummer')
    case_contact = pd.DataFrame({'n_attempts': _g.size(),
                                 'reached': _g['_reached'].any()})
    _succ = contacts[contacts['_reached'] == True]
    case_contact = case_contact.join(
        _succ.groupby('Meldingnummer')['Datum contactpoging'].min().rename('first_reach_date'))
    case_contact = case_contact.reset_index()
    # method that actually achieved contact = first SUCCESSFUL attempt's method
    # (the modal attempted method would be dominated by letters, which almost never reach)
    _sm = _succ.dropna(subset=['Wijze van contactpoging']).sort_values('Datum contactpoging')
    case_method = (_sm.groupby('Meldingnummer')['Wijze van contactpoging']
                   .first().rename('reach_method').reset_index())
else:
    case_contact = pd.DataFrame(columns=['Meldingnummer', 'n_attempts', 'reached', 'first_reach_date'])
    case_method  = pd.DataFrame(columns=['Meldingnummer', 'reach_method'])

# help decision + final result: take the last non-blank from Eindresultaat (final word)
def _last_nonblank(s):
    s = s.dropna()
    return s.iloc[-1] if len(s) else np.nan
eind = extra_tabs.get('Eindresultaat', pd.DataFrame())
if len(eind):
    _e = eind.sort_values('Datum contactpoging').groupby('Meldingnummer')
    case_eind = pd.DataFrame({'wil_hulp': _e['Wil de klant hulp'].agg(_last_nonblank),
                              'eindresultaat': _e['Wat is het eindresultaat'].agg(_last_nonblank)}).reset_index()
else:
    case_eind = pd.DataFrame(columns=['Meldingnummer', 'wil_hulp', 'eindresultaat'])

# ── Closed cases -> master survival frame ─────────────────────────────────────
_mc = meldingen[['Meldingnummer', 'Afgerond d.d.']].copy()
_mc['afgerond'] = ja_nee(meldingen['Afgerond'])
_mc = _mc[(_mc['afgerond'] == True) & _mc['Afgerond d.d.'].notna()].rename(columns={'Afgerond d.d.': 'close_date'})

eff = _mc.merge(_first[['Meldingnummer', 'Hash', 'signal_date', 'doorgezet_bool',
                        'recidive_hist', 'Type melder', 'debt_band', 'age_band', 'Leeftijd_clean']],
                on='Meldingnummer', how='inner')

for _bc in ['debt_band', 'age_band']:
    if _bc in eff.columns and hasattr(eff[_bc], 'cat'):
        eff[_bc] = eff[_bc].cat.remove_unused_categories()

# next signal from the same Hash strictly after close_date (searchsorted per household)
_allsig = signalen.dropna(subset=['Hash', 'Datum melding'])
_sig_by_hash = _allsig.sort_values('Datum melding').groupby('Hash')['Datum melding'].apply(list).to_dict()
def _next_sig(h, c):
    arr = _sig_by_hash.get(h)
    if not arr:
        return pd.NaT
    i = bisect.bisect_right(arr, c + pd.Timedelta(days=MIN_GAP_DAYS))
    return arr[i] if i < len(arr) else pd.NaT
eff['next_signal_date'] = [_next_sig(h, c) for h, c in zip(eff['Hash'], eff['close_date'])]

for _dc in ['close_date', 'signal_date', 'next_signal_date']:
    if _dc in eff.columns:
        eff[_dc] = pd.to_datetime(eff[_dc], errors='coerce')

# event=0 means no return observed: retain the case as signal-free through the data cutoff,
# then administratively right-censor it (it is not dropped or marked lost to follow-up).
eff['event'] = eff['next_signal_date'].notna().astype(int)
_end = eff['next_signal_date'].fillna(EXPORT_DATE)
eff['duration'] = (_end - eff['close_date']).dt.days
eff = eff[eff['duration'] > 0].copy()               # closed before export; positive follow-up

# ── Signal-level survival frame (for A4: non-forwarded signals never become cases) ──
_ss = signalen_x.dropna(subset=['Hash', 'Datum melding']).copy()
_ss['signal_date'] = pd.to_datetime(_ss['Datum melding'], errors='coerce')
_ss = _ss.dropna(subset=['signal_date'])
def _next_after(h, t):
    arr = _sig_by_hash.get(h)
    if not arr:
        return pd.NaT
    j = bisect.bisect_right(arr, t + pd.Timedelta(days=MIN_GAP_DAYS))
    return arr[j] if j < len(arr) else pd.NaT
_ss['next_date'] = [_next_after(h, t) for h, t in zip(_ss['Hash'], _ss['signal_date'])]
_ss['event'] = _ss['next_date'].notna().astype(int)
_ss['duration'] = (_ss['next_date'].fillna(EXPORT_DATE) - _ss['signal_date']).dt.days
sigsurv = _ss[_ss['duration'] > 0].copy()
for _bc in ['debt_band', 'age_band']:
    if _bc in sigsurv.columns and hasattr(sigsurv[_bc], 'cat'):
        sigsurv[_bc] = sigsurv[_bc].cat.remove_unused_categories()

# attach contact-derived case features
for _cf in [case_contact, case_eind, case_method]:
    if len(_cf):
        eff = eff.merge(_cf, on='Meldingnummer', how='left')
if 'reached' not in eff.columns:    eff['reached'] = False
if 'n_attempts' not in eff.columns: eff['n_attempts'] = np.nan
eff['reached'] = eff['reached'].fillna(False)
if 'first_reach_date' in eff.columns:
    eff['first_reach_date'] = pd.to_datetime(eff['first_reach_date'], errors='coerce')
    eff['days_to_reach'] = (eff['first_reach_date'] - eff['signal_date']).dt.days
else:
    eff['days_to_reach'] = np.nan

# ── Observed prior history (possible now that the export spans 2021-2026) ─────
def _prior_sigs(h, t):
    arr = _sig_by_hash.get(h)
    if not arr or pd.isna(t):
        return 0
    return bisect.bisect_left(arr, t - pd.Timedelta(days=MIN_GAP_DAYS))
eff['prior_sigs'] = [_prior_sigs(h, t) for h, t in zip(eff['Hash'], eff['signal_date'])]
eff['prior_band'] = pd.cut(eff['prior_sigs'], bins=[-1, 0, 2, 5, 1e9],
                           labels=['0', '1-2', '3-5', '6+'])
eff['prior_band'] = eff['prior_band'].cat.remove_unused_categories()

# ── Arm assignment (short codes -> labels) ────────────────────────────────────
ARM_LABEL = {'A_not_reached': 'A - not reached', 'B_refused': 'B - help refused',
             'C_accepted': 'C - help accepted',
             'B1_resolving': 'B1 - declined (already resolving)',
             'B2_difficulty': 'B2 - declined (still in difficulty)'}
_ACCEPT = {'ja', 'vervolg afspraak gemaakt', 'wenst contact met vst'}
def _arm(row):
    if not bool(row['reached']):
        return 'A_not_reached'
    hulp = str(row.get('wil_hulp', '') or '').strip().lower()
    return 'C_accepted' if hulp in _ACCEPT else 'B_refused'
def _arm_clean(row):
    a = row['arm']
    if a != 'B_refused':
        return a
    eind = str(row.get('eindresultaat', '') or '').lower()
    if any(k in eind for k in ['betaald', 'betalingsregeling', 'zelf probeert', 'quick fix', 'schuldhulp']):
        return 'B1_resolving'
    return 'B2_difficulty'
if len(eff):
    eff['arm'] = eff.apply(_arm, axis=1)
    eff['arm_clean'] = eff.apply(_arm_clean, axis=1)
else:
    eff['arm'] = pd.Series(dtype=object)
    eff['arm_clean'] = pd.Series(dtype=object)

# ── Shared survival helpers ───────────────────────────────────────────────────
def cuminc_at(dur, evt, day):
    dur = np.asarray(dur, float); evt = np.asarray(evt, int)
    if len(dur) < 5 or evt.sum() == 0:
        return np.nan
    if (dur >= day).sum() < 10:   # KM only carries its last value forward here — not a real estimate
        return np.nan
    sf = SurvfuncRight(dur, evt)
    idx = np.searchsorted(sf.surv_times, day, side='right') - 1
    if idx < 0:
        return 0.0
    return float(1 - sf.surv_prob[idx])

def cuminc_str(dur, evt, day, width=7):
    v = cuminc_at(dur, evt, day)
    return f'{v*100:{width}.1f}%' if pd.notna(v) else f'{"n/a":>{width}} '

def cox_hr(data, formula, covars, event_col='event', dur_col='duration'):
    d = data.dropna(subset=[dur_col, event_col] + covars).copy()
    if len(d) < 40 or int(d[event_col].sum()) < 5:
        print('  (Cox skipped - too few cases/events)'); return None
    try:
        r = PHReg.from_formula(formula, status=d[event_col].values, data=d).fit()
        _names = list(r.model.exog_names)
        _params = np.asarray(r.params); _pvals = np.asarray(r.pvalues)
        _ci = np.asarray(r.conf_int())
        return pd.DataFrame({'HR': np.exp(_params), 'CI2.5%': np.exp(_ci[:, 0]),
                             'CI97.5%': np.exp(_ci[:, 1]), 'p': _pvals},
                            index=_names).round(3)
    except Exception as _e:
        print(f'  Cox failed: {_e}'); return None

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"Closed, linked cases in effectiveness frame: {len(eff):,}")
if len(eff):
    print(f"  returned (event):        {int(eff['event'].sum()):,} ({eff['event'].mean()*100:.1f}%)")
    print(f"  no return observed through cutoff: {int((eff['event']==0).sum()):,}")
    print("    (retained as signal-free during observed follow-up; administratively right-censored thereafter)")
    print(f"  median follow-up (days): {eff['duration'].median():.0f}")
    print(f"  reached (any contact):   {int(eff['reached'].sum()):,} ({eff['reached'].mean()*100:.1f}%)")
    print("  arm sizes:")
    for _c, _n in eff['arm'].value_counts().items():
        print(f"    {ARM_LABEL.get(_c, _c):<34} {_n:,}")
    eff['_close_yr'] = eff['close_date'].dt.year
    if eff['_close_yr'].nunique() > 1:
        print("  90-day return by case-close year (censoring-fair; n/a = window not yet observable):")
        for _yy, _gy in eff.groupby('_close_yr'):
            if len(_gy) >= 50:
                print(f"    {int(_yy)}: {cuminc_str(_gy['duration'], _gy['event'], 90, 5)}  (n={len(_gy):,})")
else:
    print("  No closed+linked cases — check the export (needs Meldingnummer on signals + close dates).")

### 9a. Experiment A1+A2 — Does contact help even when help is refused?

Closed cases are split into three arms: **A** never reached, **B** reached but declined help,
**C** reached and accepted help. Each arm gets a time-to-return curve plus an adjusted comparison.

The key contrast is **B vs A**: if households that declined help still return less than households
never reached, then contact *itself* has value — a "wake-up" effect. **A2** re-runs the test after
removing refusers who were already solving it themselves (they would make B look good for the
wrong reason).


In [ ]:
_report_section('9. Contact Decomposition - Does Contact Reduce Return?')

_arms = ['A_not_reached', 'B_refused', 'C_accepted']
_present = [a for a in _arms if (eff['arm'] == a).sum() > 0]

if len(eff) == 0 or int(eff['event'].sum()) == 0 or len(_present) < 2:
    print('No return events or fewer than two observed arms to compare.')
else:
    fig, ax = plt.subplots(figsize=(11, 6))
    for a in _present:
        g = eff[eff['arm'] == a]
        sf = SurvfuncRight(g['duration'].values, g['event'].values)
        ax.step(sf.surv_times, (1 - sf.surv_prob) * 100, where='post',
                linewidth=2, label=f'{ARM_LABEL[a]} (n={len(g)})')
    ax.set_xlim(0, 730)
    ax.set_xlabel('Days since case close'); ax.set_ylabel('Cumulative return %')
    ax.set_title('Return after case close, by contact arm')
    ax.legend()
    plt.tight_layout(); plt.show()

    _HZ = [30, 90, 180, 270, 365, 540, 730]
    print('Cumulative return % by arm (values behind the curves):')
    print('  ' + 'arm'.ljust(20) + ''.join(f'{h}d'.rjust(8) for h in _HZ) + '       n')
    for a in _present:
        g = eff[eff['arm'] == a]
        print('  ' + ARM_LABEL[a].ljust(20)
              + ''.join(cuminc_str(g['duration'], g['event'], h) for h in _HZ)
              + f'{len(g):8d}')
    print('  (n/a = fewer than 10 cases still under observation at that horizon)')
    _small = [ARM_LABEL[a] for a in _present if (eff['arm'] == a).sum() < 50]
    if _small:
        print(f"  (small arm{'s' if len(_small) > 1 else ''} — interpret with caution: {', '.join(_small)})")

    try:
        _sub = eff[eff['arm'].isin(_present)]
        _chi, _p = survdiff(_sub['duration'].values, _sub['event'].values, _sub['arm'].values)
        print(f'\nLog-rank across arms: chi2={_chi:.2f}, p={_p:.4f}')
    except Exception as _e:
        print(f'Log-rank failed: {_e}')

    print('\nAdjusted hazard ratios (ref = A not reached; HR<1 = fewer/slower returns):')
    hr = cox_hr(eff, 'duration ~ C(arm, Treatment(reference="A_not_reached")) + recidive_hist '
                     '+ C(prior_band) + C(debt_band) + C(age_band)',
                ['arm', 'recidive_hist', 'prior_band', 'debt_band', 'age_band'])
    if hr is not None:
        display(hr)

    # ── A2: clean contrast — refusers still in difficulty (B2) vs not reached (A) ──
    print('\nA2 - Clean test (excludes B1 already-resolving refusers):')
    for a in ['A_not_reached', 'B2_difficulty', 'C_accepted']:
        g = eff[eff['arm_clean'] == a]
        if len(g) > 0:
            print(f'  {ARM_LABEL[a]:<34} 90d return={cuminc_str(g["duration"], g["event"], 90, 5)} (n={len(g)})')

    if eff['close_date'].dt.year.nunique() > 1:
        _yrows = []
        for _yy, _gy in eff.groupby(eff['close_date'].dt.year):
            _r = {'year': int(_yy)}
            for a in _present:
                _gg = _gy[_gy['arm'] == a]
                _v = cuminc_at(_gg['duration'], _gg['event'], 90) if len(_gg) > 0 else np.nan
                _r[ARM_LABEL[a]] = round(_v * 100, 1) if pd.notna(_v) else np.nan
            _yrows.append(_r)
        print('\n90-day return % by arm and close year (all observed groups; NaN = insufficient follow-up):')
        print(pd.DataFrame(_yrows).set_index('year').to_string())

## 10. Experiments A3 + A4 + A5 — Method, Forwarding, Resolution

Three questions about what happens *after* the funnel:

- **A3:** Divosa ranks contact methods by *reach* (phone > home visit > email > sms). Does the
  best-reaching method also best *prevent return*? Full time-to-return curves and fixed-horizon
  return rates are compared by the first successful contact method. If the rankings diverge,
  optimising reach ≠ optimising effect.
- **A4:** do forwarded signals (`Doorgezet = Ja`) lead to fewer new signals than non-forwarded ones?
- **A5:** which final case results (paid, payment arrangement, no contact…) predict *not* returning?


In [ ]:
_report_section('10. Reach vs Effectiveness & Resolution Quality')

# ── A4 - forwarding -> next signal (signal-level; non-forwarded signals never become cases) ──
print('A4 - Forwarding (Doorgezet) -> next signal  [signal-level Cox HR, ref = not forwarded]:')
hr = cox_hr(sigsurv, 'duration ~ doorgezet_bool + recidive_hist', ['doorgezet_bool', 'recidive_hist'])
if hr is not None:
    display(hr)

# ── A3 - does the best-reaching method also best prevent return? ──────────────
print('\nA3 - Return by the method that achieved contact (first successful attempt; all observed methods):')
if 'reach_method' in eff.columns:
    _m = eff[eff['reached'] == True].dropna(subset=['reach_method'])
    _method_groups = list(_m.groupby('reach_method'))
    _rows = []
    for meth, g in _method_groups:
        _rows.append((str(meth)[:45], len(g), cuminc_at(g['duration'], g['event'], 90) * 100,
                      cuminc_at(g['duration'], g['event'], 365) * 100))
    if _rows:
        _mt = pd.DataFrame(_rows, columns=['method', 'n', 'return_90d_%', 'return_365d_%']).sort_values('return_90d_%')
        display(_mt.round(1))

        # Same survival-derived cumulative-return graph as §9a, now stratified by reach method.
        fig, ax = plt.subplots(figsize=(11, 6))
        for _i, (meth, g) in enumerate(_method_groups):
            sf = SurvfuncRight(g['duration'].values, g['event'].values)
            ax.step(sf.surv_times, (1 - sf.surv_prob) * 100, where='post',
                    linewidth=2, color=COLORS[_i % len(COLORS)],
                    label=f'{str(meth)[:45]} (n={len(g)})')
        ax.set_xlim(0, 730)
        ax.set_xlabel('Days since case close'); ax.set_ylabel('Cumulative return %')
        ax.set_title('Return after case close, by method that achieved contact')
        ax.legend(fontsize=9)
        plt.tight_layout(); plt.show()

        fig, ax = plt.subplots(figsize=(11, 5))
        ax.barh(_mt['method'], _mt['return_90d_%'], color=COLORS[0])
        ax.set_xlabel('90-day return %  (lower = more effective at preventing return)')
        ax.set_title('A3 - Return by method that achieved contact')
        plt.tight_layout(); plt.show()
        print('Compare this ordering with Divosa reach ranking (phone > home-visit > email > sms):')
        print('divergence => reach-effective method is NOT the same as return-preventing method.')
        if _m['close_date'].dt.year.nunique() > 1:
            _yr3 = []
            for (_mm, _yy), _gg in _m.groupby(['reach_method', _m['close_date'].dt.year]):
                _v = cuminc_at(_gg['duration'], _gg['event'], 90)
                _yr3.append((str(_mm)[:34], int(_yy), len(_gg),
                             round(_v * 100, 1) if pd.notna(_v) else np.nan))
            if _yr3:
                _d3 = pd.DataFrame(_yr3, columns=['method', 'year', 'n', 'ret90'])
                _p3 = _d3.pivot(index='method', columns='year', values='ret90')
                _n3 = _d3.pivot(index='method', columns='year', values='n')
                print('\nA3 per year - 90d return by reaching method (all observed groups):')
                print(_p3.to_string())
                print('\nCase counts behind the method-by-year estimates:')
                print(_n3.to_string())
    else:
        print('  No reached case has a recorded successful contact method.')
else:
    print('  primary_method unavailable.')

# ── A5 - does a good resolution predict not returning? ────────────────────────
print('\nA5 - 90-day return by final result (Eindresultaat, n>=30):')
if 'eindresultaat' in eff.columns:
    _rows = []
    for res, g in eff.dropna(subset=['eindresultaat']).groupby('eindresultaat'):
        if len(g) >= 30:
            _rows.append((str(res)[:55], len(g), cuminc_at(g['duration'], g['event'], 90) * 100,
                          cuminc_at(g['duration'], g['event'], 365) * 100))
    if _rows:
        display(pd.DataFrame(_rows, columns=['eindresultaat', 'n', 'return_90d_%', 'return_365d_%'])
                .sort_values('return_90d_%').round(1))
    else:
        print('  No result category with >=30 cases.')
else:
    print('  eindresultaat unavailable.')

## 11. Experiment B — Who Returns? (Targeting)

Profiles the repeat-signal rate by reporter type, debt band, age band and minor children,
to answer *who should be prioritised*. Uses the source system's own `Recidive` flag as ground
truth; the full 2021–2026 history in the export (§13 episodes) provides a consistent
cross-check. Groups with fewer than 30 signals are hidden.


In [ ]:
_report_section('11. Targeting - Who Returns? (built-in Recidive flag)')
MIN_CELL = 30
obsB = signalen_x.copy()
obsB['recidive_hist'] = ja_nee(obsB['Recidive'])
_ov = obsB['recidive_hist'].mean() * 100
print(f"Overall built-in recidive rate: {_ov:.1f}%  (n={int(obsB['recidive_hist'].notna().sum()):,})")

_dims = [('Type melder', 'Reporter type'), ('debt_band', 'Debt band'),
         ('age_band', 'Age band'), ('Minderjarige kinderen', 'Minor children')]

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
for ax, (col, label) in zip(axes.flatten(), _dims):
    if col not in obsB.columns:
        ax.text(0.5, 0.5, f'{label}\nnot available', ha='center', va='center', transform=ax.transAxes); continue
    g = obsB.dropna(subset=[col]).groupby(col)['recidive_hist'].agg(n='count', rate='mean')
    g = g[g['n'] >= MIN_CELL].sort_values('rate')
    if len(g) == 0:
        ax.text(0.5, 0.5, f'{label}\ntoo few', ha='center', va='center', transform=ax.transAxes); continue
    ax.barh(g.index.astype(str), g['rate'] * 100, color=COLORS[1])
    ax.axvline(_ov, color='red', linestyle='--', linewidth=1.5, label=f'overall {_ov:.1f}%')
    ax.xaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_title(f'Recidive by {label}'); ax.set_xlabel('Repeat-signal %'); ax.legend(fontsize=8)
plt.suptitle('Targeting - repeat-signal rate by household profile', y=1.01)
plt.tight_layout(); plt.show()

print('\nHighest-risk segment per dimension:')
for col, label in _dims:
    if col not in obsB.columns:
        continue
    g = obsB.dropna(subset=[col]).groupby(col)['recidive_hist'].agg(n='count', rate='mean')
    g = g[g['n'] >= MIN_CELL].sort_values('rate', ascending=False)
    if len(g):
        print(f'  {label:<16}: {g.index[0]}  ->  {g.iloc[0]["rate"]*100:.1f}%  (n={int(g.iloc[0]["n"]):,})')

print('\nFull recidive breakdown by dimension (values behind the bars, n>=30):')
for col, label in _dims:
    if col not in obsB.columns:
        continue
    g = obsB.dropna(subset=[col]).groupby(col)['recidive_hist'].agg(n='count', rate='mean')
    g = g[g['n'] >= MIN_CELL].sort_values('rate', ascending=False)
    if len(g):
        g['recidive_%'] = (g['rate'] * 100).round(1)
        print(f'\n- {label}:')
        print(g.drop(columns='rate').to_string())

_yc = obsB.dropna(subset=['Datum melding']).copy()
_yc['_yr'] = _yc['Datum melding'].dt.year
if _yc['_yr'].nunique() > 1:
    _yc['_rec'] = ja_nee(_yc['Recidive']) == True
    for _col, _label in _dims:
        if _col not in _yc.columns:
            continue
        _pv11 = _yc.pivot_table(index=_col, columns='_yr', values='_rec',
                                aggfunc='mean', observed=True) * 100
        _nv11 = _yc.pivot_table(index=_col, columns='_yr', values='_rec',
                                aggfunc='count', observed=True)
        _pv11 = _pv11.where(_nv11 >= 30).dropna(how='all')
        if len(_pv11):
            print(f'\nRecidive % by {_label} per year (cells n>=30):')
            print(_pv11.round(1).to_string())

## 12. Experiments C1–C4 — Return Timing, Contact Effort & Recurrence Patterns

Is recorded contact effort associated with subsequent return?
Four questions about *when* and *how much*:

- **C3:** when do households return after closure — first weeks or a long tail?
- **C1:** does reaching a household *faster* reduce its chance of returning?
- **C2:** does the *number* of contact attempts relate to returning?
- **C4:** how many signals does a household send — one-off or chronic returners?

For C1, the descriptive table defines **Fast** as contact at or below the observed median number
of days to reach and **Slow** as contact above that median. These are text labels, not 0/1
codes. The adjusted Cox model uses the actual number of days to reach as a continuous variable.


In [ ]:
_report_section('12. Return Timing, Contact Effort & Recurrence Patterns')

# ── C3 - overall recurrence timing ────────────────────────────────────────────
if len(eff) and int(eff['event'].sum()) >= 10:
    sf = SurvfuncRight(eff['duration'].values, eff['event'].values)
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.step(sf.surv_times, (1 - sf.surv_prob) * 100, where='post', color=COLORS[2], linewidth=2)
    ax.set_xlim(0, 730); ax.set_xlabel('Days since case close'); ax.set_ylabel('Cumulative return %')
    ax.set_title('C3 - When do households return? (overall)')
    plt.tight_layout(); plt.show()
    _HZ = [30, 90, 180, 270, 365, 540, 730]
    print("Overall cumulative return % (values behind the curve):")
    print('  ' + ''.join(f'{h}d'.rjust(8) for h in _HZ))
    print('  ' + ''.join(cuminc_str(eff['duration'], eff['event'], h) for h in _HZ))
    print('  (n/a = fewer than 10 cases still under observation at that horizon)')
else:
    print('C3 - Not enough events for a recurrence curve.')

# ── C1 - speed to contact ─────────────────────────────────────────────────────
print('\nC1 - Speed-to-contact -> return:')
if 'days_to_reach' in eff.columns and eff['days_to_reach'].notna().any():
    _r = eff[(eff['reached'] == True) & eff['days_to_reach'].notna()].copy()
    if len(_r) >= 40:
        _med = _r['days_to_reach'].median()
        _r['speed_grp'] = np.where(_r['days_to_reach'] <= _med, 'Fast', 'Slow')
        print(f'  Descriptive split at the observed median: {_med:g} days')
        print(f'    Fast = days_to_reach <= {_med:g}; Slow = days_to_reach > {_med:g}')
        print('    Fast/Slow are text labels, not binary 0/1 values.')
        _speed_rows = []
        for grp in ['Fast', 'Slow']:
            g = _r[_r['speed_grp'] == grp]
            _criterion = (f'<= {_med:g} days' if grp == 'Fast' else f'> {_med:g} days')
            _speed_rows.append((grp, _criterion, len(g), g['days_to_reach'].median(),
                                cuminc_at(g['duration'], g['event'], 90) * 100,
                                cuminc_at(g['duration'], g['event'], 365) * 100))
        display(pd.DataFrame(_speed_rows, columns=['group', 'definition', 'n',
                                                   'median_days_to_reach',
                                                   'return_90d_%', 'return_365d_%']).round(1))
        print('\n  Adjusted Cox model: days_to_reach enters as a continuous variable, not as Fast=0/1.')
        print('  Its hazard ratio is interpreted per one additional day before successful contact.')
        hr = cox_hr(_r, 'duration ~ days_to_reach + recidive_hist', ['days_to_reach', 'recidive_hist'])
        if hr is not None:
            display(hr)
    else:
        print('  Too few reached cases with contact timing.')
else:
    print('  days_to_reach unavailable.')

# ── C2 - contact-attempt intensity (descriptive association) ──────────────────
print('\nC2 - Number of attempts -> return:')
if 'n_attempts' in eff.columns and eff['n_attempts'].notna().any():
    _a = eff.dropna(subset=['n_attempts']).copy()
    _a['att_band'] = pd.cut(_a['n_attempts'], bins=[0, 1, 3, 6, 1e9], labels=['1', '2-3', '4-6', '7+'])
    _rows = []
    for b, g in _a.groupby('att_band'):
        if len(g) >= 30:
            _rows.append((str(b), len(g), cuminc_at(g['duration'], g['event'], 90) * 100,
                          cuminc_at(g['duration'], g['event'], 365) * 100))
    if _rows:
        display(pd.DataFrame(_rows, columns=['attempts', 'n', 'return_90d_%', 'return_365d_%']).round(1))
    else:
        print('  Too few cases per attempt band.')
else:
    print('  n_attempts unavailable.')

# ── C4 - recurrent signals per household ──────────────────────────────────────
print('\nC4 - Repeated signals per household:')
_sph = signalen.dropna(subset=['Hash']).groupby('Hash').size()
if len(_sph):
    print(f'  households={len(_sph):,}  mean={_sph.mean():.2f}  median={_sph.median():.0f}  '
          f'p99={_sph.quantile(.99):.0f}  with >1 signal: {(_sph > 1).mean()*100:.1f}%')

## 13. Module 1 — Causal Upgrade (E1, E2, E3, G4)

Round-1 results showed selection bias: worse-off households are reached and accept help more
often, so between-household comparisons are misleading. This module compares households
**with themselves** across their own episodes, and stress-tests the outcome definition.
The prep cell below collapses each household's signals into **episodes** (signals within
`MIN_GAP_DAYS` of each other are the same episode) — the shared unit for §13–§15.


In [ ]:
_report_section('13. Module 1 - Causal Upgrade: preparation')
# ── Episode table: collapse each household's signals into episodes ────────────
_t = signalen_x.dropna(subset=['Hash', 'Datum melding']).sort_values(['Hash', 'Datum melding']).copy()
_t['_prev'] = _t.groupby('Hash')['Datum melding'].shift()
_t['_newep'] = (_t['_prev'].isna()) | ((_t['Datum melding'] - _t['_prev']).dt.days > MIN_GAP_DAYS)
_t['epnum'] = _t.groupby('Hash')['_newep'].cumsum()

epi = (_t.groupby(['Hash', 'epnum'])
       .agg(start=('Datum melding', 'min'), end=('Datum melding', 'max'),
            n_signals=('SignaalID', 'count'))
       .reset_index())
_types = (_t.dropna(subset=['Type melder']).sort_values('Datum melding')
          .groupby(['Hash', 'epnum'])['Type melder'].first().rename('ep_type'))
epi = epi.merge(_types, on=['Hash', 'epnum'], how='left').sort_values(['Hash', 'epnum'])
epi['next_start'] = epi.groupby('Hash')['start'].shift(-1)
epi['gap_days']  = (epi['next_start'] - epi['end']).dt.days
epi['gap_event'] = epi['next_start'].notna().astype(int)
epi['gap_dur']   = np.where(epi['gap_event'] == 1, epi['gap_days'],
                            (EXPORT_DATE - epi['end']).dt.days)
epi = epi[epi['gap_dur'] > 0].copy()

_eph = epi.groupby('Hash').size()
print(f"Episodes: {len(epi):,} across {epi['Hash'].nunique():,} households "
      f"(signals <= {MIN_GAP_DAYS}d apart merged into one episode)")
print(f"Episodes per household: mean={_eph.mean():.2f}  median={_eph.median():.0f}  p99={_eph.quantile(.99):.0f}")
_eyr = epi['start'].dt.year.value_counts().sort_index()
if len(_eyr) > 1:
    print("Episodes started per year:")
    print(_eyr.rename('episodes').to_string())

# ── Creditor breadth at case start (shared by E1b and G2) ─────────────────────
_ts2 = signalen_x.dropna(subset=['Hash', 'Datum melding', 'Type melder']).sort_values('Datum melding')
_by_h = {h: list(zip(g['Datum melding'].tolist(), g['Type melder'].tolist()))
         for h, g in _ts2.groupby('Hash')}
def _breadth(h, t):
    lst = _by_h.get(h)
    if not lst or pd.isna(t):
        return np.nan
    return len({ty for d0, ty in lst if d0 <= t})
eff['breadth'] = [_breadth(h, t) for h, t in zip(eff['Hash'], eff['signal_date'])]
eff['breadth_band'] = pd.cut(eff['breadth'], bins=[0, 1, 2, 99], labels=['1', '2', '3+'])

# ── Case-level episode numbering for within-household models ──────────────────
eff_sorted = eff.sort_values(['Hash', 'close_date']).copy()
eff_sorted['case_epnum'] = eff_sorted.groupby('Hash').cumcount() + 1
eff_multi = eff_sorted[eff_sorted.groupby('Hash')['Hash'].transform('size') >= 2].copy()
print(f"Households with >=2 closed cases (usable for within-household designs): "
      f"{eff_multi['Hash'].nunique():,} ({len(eff_multi):,} cases)")


### 13a. E1 — Within-Household Comparison

Compares the **same household** across its own episodes: does the episode where it was
reached (or accepted help) show a lower risk of a new signal than its other episodes?
Stratified Cox with household as stratum removes all stable household confounding — the
strongest available answer to the Round-1 selection problem. Only "discordant" households
(exposure varies across their episodes) contribute.

**E1b** repeats the model adjusted for episode-level severity (episode number, prior
signals, creditor breadth, debt band, year); stable household traits are already absorbed
by the stratum. If the effect moves toward 1, the unadjusted result was partly severity.


In [ ]:
_report_section('13a. E1 - Within-Household Comparison')

def _within_cox(d, expo, label):
    _disc = d.groupby('Hash')[expo].nunique()
    dd = d[d['Hash'].isin(_disc[_disc > 1].index)].copy()
    n_hh = dd['Hash'].nunique()
    print(f"{label}: discordant households = {n_hh:,}  "
          f"(episodes {len(dd):,}, events {int(dd['event'].sum()):,})")
    if len(dd) < 100 or dd['event'].sum() < 30:
        print('  Too few discordant episodes/events - model skipped.')
        return
    try:
        r = PHReg.from_formula(f'duration ~ {expo}', data=dd,
                               status=dd['event'].values, strata=dd['Hash'].values).fit()
        _p = np.asarray(r.params); _ci = np.asarray(r.conf_int()); _pv = np.asarray(r.pvalues)
        display(pd.DataFrame({'HR': np.exp(_p), 'CI2.5%': np.exp(_ci[:, 0]),
                              'CI97.5%': np.exp(_ci[:, 1]), 'p': _pv},
                             index=r.model.exog_names).round(3))
    except Exception as _e:
        print(f'  stratified Cox failed: {_e}')

if len(eff_multi) == 0:
    print('No households with multiple closed cases - E1 skipped.')
else:
    print('Inclusion flow (who this analysis describes):')
    print(f'  households with >=1 closed, linked case: {eff["Hash"].nunique():,}')
    print(f'  with >=2 closed cases:                   {eff_multi["Hash"].nunique():,}  '
          f'({len(eff_multi):,} cases)')
    print('  (only households whose exposure VARIES across episodes identify the effect;')
    print('   their counts are printed per exposure below)\n')
    eff_multi['reached_f'] = eff_multi['reached'].astype(float)
    eff_multi['accepted']  = (eff_multi['arm'] == 'C_accepted').astype(float)
    _within_cox(eff_multi, 'reached_f', 'Exposure = reached in that episode')
    print()
    _within_cox(eff_multi, 'accepted', 'Exposure = help accepted in that episode')
    print('\nHR < 1 = the SAME household returns less after exposed episodes.')

    # ── E1b: adjusted for episode-level (time-varying) severity ───────────────
    print('\nE1b - same model, adjusted for how severe each episode was (episode number,')
    print('prior signals, creditor breadth, debt band, year). Stable household traits are')
    print('already absorbed by the household stratum, so only episode-varying ones enter.')
    _dsc = eff_multi.groupby('Hash')['reached_f'].nunique()
    db = eff_multi[eff_multi['Hash'].isin(_dsc[_dsc > 1].index)].copy()
    db['yr'] = db['close_date'].dt.year - int(db['close_date'].dt.year.min())
    db = db.dropna(subset=['duration', 'event', 'debt_band'])
    if hasattr(db['debt_band'], 'cat'):
        db['debt_band'] = db['debt_band'].cat.remove_unused_categories()
    if len(db) >= 120 and db['event'].sum() >= 40:
        _terms = ['reached_f']
        for _tv in ['case_epnum', 'prior_sigs', 'breadth', 'yr']:
            if _tv in db.columns and db[_tv].nunique() > 1:
                _terms.append(_tv)
        if db['debt_band'].nunique() > 1:
            _terms.append('C(debt_band)')
        try:
            r = PHReg.from_formula('duration ~ ' + ' + '.join(_terms), data=db,
                                   status=db['event'].values, strata=db['Hash'].values).fit()
            _p = np.asarray(r.params); _ci = np.asarray(r.conf_int()); _pv = np.asarray(r.pvalues)
            display(pd.DataFrame({'HR': np.exp(_p), 'CI2.5%': np.exp(_ci[:, 0]),
                                  'CI97.5%': np.exp(_ci[:, 1]), 'p': _pv},
                                 index=r.model.exog_names).round(3))
            print('Compare reached_f with the unadjusted model above: if it moves toward 1,')
            print('part of the unadjusted effect was episode severity, not contact.')
        except Exception as _e:
            print(f'  adjusted model failed: {_e}')
    else:
        print(f'  Too few complete discordant episodes for the adjusted model (n={len(db)}).')


### 13b. E2 — State Dependence vs Chronic Vulnerability

Does each episode raise the risk of the next (escalation — early intervention can break a
spiral), or do a fixed group of chronic households generate most signals (needs a different
service model)? Tested two ways: hazard of the next episode by episode number, and
within-household inter-episode gap trends.


In [ ]:
_report_section('13b. E2 - State Dependence vs Chronic Vulnerability')
_e = epi.copy()
_e['ep_band'] = pd.cut(_e['epnum'], bins=[0, 1, 2, 3, 1e9], labels=['1', '2', '3', '4+'])

print('Cumulative probability of a NEXT episode within 180d, by episode number:')
for b, g in _e.groupby('ep_band'):
    if len(g) >= 30:
        print(f'  after episode {b:>2}: 180d={cuminc_str(g["gap_dur"], g["gap_event"], 180, 5)}  '
              f'365d={cuminc_str(g["gap_dur"], g["gap_event"], 365, 5)}  (n={len(g):,})')

d = _e.dropna(subset=['ep_band']).copy()
_bandok = [b for b, g in d.groupby('ep_band') if len(g) >= 30 and g['gap_event'].sum() >= 5]
d = d[d['ep_band'].isin(_bandok)]
d['ep_band'] = d['ep_band'].cat.remove_unused_categories()
if len(d) >= 200 and d['gap_event'].sum() >= 50 and d['ep_band'].nunique() >= 2:
    try:
        r = PHReg.from_formula('gap_dur ~ C(ep_band, Treatment(reference="1"))',
                               data=d, status=d['gap_event'].values).fit()
        _p = np.asarray(r.params); _ci = np.asarray(r.conf_int()); _pv = np.asarray(r.pvalues)
        print('\nGap-time Cox - hazard of next episode by episode number (ref = 1st episode):')
        display(pd.DataFrame({'HR': np.exp(_p), 'CI2.5%': np.exp(_ci[:, 0]),
                              'CI97.5%': np.exp(_ci[:, 1]), 'p': _pv},
                             index=r.model.exog_names).round(3))
        print('NOTE: rising HRs can reflect escalation OR chronic households accumulating episodes;')
        print('the within-household gap trend below separates the two.')
    except Exception as _e2:
        print(f'Gap-time Cox failed: {_e2}')
else:
    print('Too few episodes for the gap-time model.')

# within-household gap trend (needs >=2 completed gaps = >=3 episodes)
_g = epi.dropna(subset=['gap_days'])
_cnt = _g.groupby('Hash').size()
short = long_ = flat = 0
for _h in _cnt[_cnt >= 2].index:
    gg = _g[_g['Hash'] == _h].sort_values('epnum')['gap_days'].values
    if gg[-1] < gg[0] * 0.75:   short += 1
    elif gg[-1] > gg[0] * 1.33: long_ += 1
    else:                       flat += 1
tot = short + long_ + flat
if tot >= 30:
    print(f"\nWithin-household gap trends ({tot:,} households with >=2 completed gaps):")
    print(f"  gaps shortening (escalating):  {short:,} ({short/tot*100:.1f}%)")
    print(f"  gaps lengthening (recovering): {long_:,} ({long_/tot*100:.1f}%)")
    print(f"  roughly stable:                {flat:,} ({flat/tot*100:.1f}%)")
    print('Escalation-dominant = state dependence; stable/mixed = chronic heterogeneity.')
else:
    print('\nToo few households with >=3 episodes for the gap-trend test.')


### 13c. E3 — Trajectory Around Help Acceptance (exploratory)

**Exploratory analysis — not causal proof.** Does a household's own signal rate bend after it
accepts help? t0 is the **first successful contact** of the accepted case (the actual
conversation; falls back to the close date when no contact date is recorded). Because help is
accepted at the household's worst moment, a post-t0 decline is expected from regression to the
mean alone (the "Ashenfelter dip"); the defense is the **reached-but-refused comparison group
aligned at the same moment** — the test is whether the acceptors' curve bends *more*.


In [ ]:
_report_section('13c. E3 - Trajectory Around Help Acceptance')
_BINS = [(-24, -12), (-12, -6), (-6, -3), (-3, 0), (0, 3), (3, 6), (6, 12), (12, 24)]
_OBS_START = signalen['Datum melding'].min()

def _relative_rates(t0map, label):
    rows = []
    for a, b in _BINS:
        n_hh = 0; n_sig = 0
        for _h, _t0 in t0map.items():
            w0 = _t0 + pd.Timedelta(days=int(a * 30.44))
            w1 = _t0 + pd.Timedelta(days=int(b * 30.44))
            if w1 > EXPORT_DATE or w0 < _OBS_START:
                continue                      # only households whose window is fully observed
            arr = _sig_by_hash.get(_h, [])
            n_sig += bisect.bisect_left(arr, w1) - bisect.bisect_left(arr, w0)
            n_hh += 1
        rows.append((f'{a}..{b}m', n_hh, n_sig / max(n_hh, 1) / (b - a)))
    return pd.DataFrame(rows, columns=['window', 'n_households', f'{label} signals/hh/month'])

# t0 = the actual conversation (first successful contact); fallback = close date
if 'first_reach_date' in eff_sorted.columns:
    _t0raw = pd.to_datetime(eff_sorted['first_reach_date'], errors='coerce')
else:
    _t0raw = pd.Series(pd.NaT, index=eff_sorted.index)
_n_fb = int((eff_sorted['arm'].isin(['C_accepted', 'B_refused']) & _t0raw.isna()).sum())
eff_sorted['t0'] = _t0raw.fillna(eff_sorted['close_date'])
_acc = eff_sorted[eff_sorted['arm'] == 'C_accepted'].groupby('Hash')['t0'].min()
_ref = eff_sorted[eff_sorted['arm'] == 'B_refused'].groupby('Hash')['t0'].min()
_ref = _ref[~_ref.index.isin(_acc.index)]
print(f"Acceptor households: {len(_acc):,}   comparison (reached-but-refused): {len(_ref):,}")
print(f"t0 anchor: first successful contact ({_n_fb} case(s) without one fell back to close date)")

if len(_acc) < 20:
    print('Too few help-acceptor households - E3 skipped.')
else:
    ta = _relative_rates(_acc.to_dict(), 'acceptors')
    tr = _relative_rates(_ref.to_dict(), 'refusers')
    tab = ta.merge(tr, on='window', suffixes=('', '_r'))
    print('\nSignal rate around the aligned moment (t0 = first successful contact of that case):')
    display(tab.round(3))
    fig, ax = plt.subplots(figsize=(10, 5))
    _x = range(len(_BINS))
    ax.plot(_x, ta.iloc[:, 2], marker='o', linewidth=2, color=COLORS[0], label=f'help accepted (n<= {len(_acc)})')
    ax.plot(_x, tr.iloc[:, 2], marker='s', linewidth=2, color=COLORS[1], label=f'reached, refused (n<= {len(_ref)})')
    ax.axvline(3.5, color='red', linestyle='--', alpha=0.6)
    ax.set_xticks(list(_x)); ax.set_xticklabels([w for w, _, _ in zip(ta['window'], _x, _x)])
    ax.set_xlabel('months relative to t0'); ax.set_ylabel('signals per household per month')
    ax.set_title('E3 - Own trajectory around help acceptance vs aligned refusers')
    ax.legend()
    plt.tight_layout(); plt.show()
    print('Read: both curves peaking before t0 = Ashenfelter dip (crisis-time selection).')
    print('Evidence for help = the ACCEPTOR curve bends down more than the refuser curve after t0.')


### 13d. G4 — Silent Success vs Disappearance from Observation

"No new signal" is not proof of success — the household may simply have left the observation
system. This cell profiles non-returners and adds a sharp contrast: **selective exit**
(household stopped one creditor type but still appears in others — observable resolution)
vs **total disappearance** (vanished from all types — ambiguous). Output: bounds on the true
success rate, not a point estimate.


In [ ]:
_report_section('13d. G4 - Silent Success vs Disappearance')
_nr = eff[eff['event'] == 0]
print(f"Non-returning closed cases: {len(_nr):,} of {len(eff):,} ({len(_nr)/max(len(eff),1)*100:.1f}%)")
if len(_nr) >= 30:
    print('\nProfile of non-returning cases:')
    for _col, _lab in [('eindresultaat', 'Final result'), ('reached', 'Reached'),
                       ('recidive_hist', 'Prior recidive flag')]:
        if _col in _nr.columns:
            vc = _nr[_col].astype(str).value_counts().head(6)
            print(f'- {_lab}:')
            print((vc / len(_nr) * 100).round(1).rename('%').to_string()); print()

# selective vs total exit per creditor type
_EXIT = 180
_ts = signalen_x.dropna(subset=['Hash', 'Datum melding', 'Type melder'])
_last_all  = _ts.groupby('Hash')['Datum melding'].max()
_hh_gone   = (EXPORT_DATE - _last_all).dt.days >= _EXIT
_lt = _ts.groupby(['Hash', 'Type melder'])['Datum melding'].max().reset_index()
_lt['hh_last']  = _lt['Hash'].map(_last_all)
_lt['selective'] = (_lt['hh_last'] - _lt['Datum melding']).dt.days >= _EXIT
_lt['hh_gone']   = _lt['Hash'].map(_hh_gone)
rows = []
for _ty, g in _lt.groupby('Type melder'):
    if len(g) < 30:
        continue
    sel  = int(g['selective'].sum())
    tot_ = int((~g['selective'] & g['hh_gone']).sum())
    rows.append((str(_ty)[:28], len(g), sel, tot_,
                 round(sel / max(sel + tot_, 1) * 100, 1)))
if rows:
    print(f"Exits per creditor type (exit = no signal of that type for >={_EXIT}d):")
    display(pd.DataFrame(rows, columns=['type', 'household-type pairs', 'selective exit',
                                        'total disappearance', 'selective %']))
    print('Selective exit = stopped this type, still visible in others -> observable resolution.')
    print('Total disappearance = gone from ALL types -> resolved OR left observation (ambiguous).')
else:
    print('Too few household-type pairs for the exit analysis.')


## 14. Module 2 — Trajectory & Early-Warning Science (G1, G2, G3, G5)

Tests the founding premise of Vroegsignalering (are some signals genuinely *early*?) and maps
the recurring population. Uses the episode table from §13. All "first" statements mean *first
observed since 2021* (Wgs start) — earlier history is not visible.


### 14a. G1 — Debt Cascade: Transitions Between Creditor Types

Households in trouble choose what to stop paying first. The within-household transition matrix
P(next episode type | current type) reveals a payment-priority ladder; the first-mover index
shows which types open a household's observed history. Types that systematically come first
are true early-warning signals.

**G1b** decomposes every transition into *same type again*, *known other type* and
*new type never seen before* — separating unresolved re-reporting from the true cascade.


In [ ]:
_report_section('14a. G1 - Debt Cascade: Transitions Between Creditor Types')
_seq = epi.dropna(subset=['ep_type']).sort_values(['Hash', 'epnum']).copy()
_seq['next_type'] = _seq.groupby('Hash')['ep_type'].shift(-1)
_tr = _seq.dropna(subset=['next_type'])
if len(_tr) < 100:
    print(f'Too few episode transitions ({len(_tr)}) for a stable matrix.')
else:
    mat = pd.crosstab(_tr['ep_type'], _tr['next_type'], normalize='index') * 100
    _rowcnt = _tr['ep_type'].value_counts()
    mat = mat.loc[[i for i in mat.index if _rowcnt.get(i, 0) >= 30]]
    print(f"Episode-to-episode transitions: {len(_tr):,}  (rows with n>=30 shown)")
    print('P(next episode type | current type), % per row:')
    display(mat.round(1))

    fig, ax = plt.subplots(figsize=(9, max(4, 0.8 * len(mat))))
    im = ax.imshow(mat.values, cmap='Blues', aspect='auto')
    ax.set_xticks(range(len(mat.columns))); ax.set_xticklabels([str(c)[:18] for c in mat.columns], rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(len(mat.index)));  ax.set_yticklabels([str(i)[:18] for i in mat.index], fontsize=8)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            ax.text(j, i, f'{mat.values[i, j]:.0f}', ha='center', va='center', fontsize=8,
                    color='white' if mat.values[i, j] > 50 else 'black')
    ax.set_title('G1 - P(next episode type | current type) %')
    plt.tight_layout(); plt.show()

    _first = _seq.groupby('Hash')['ep_type'].first()
    fm = _first.value_counts(); base = _seq['ep_type'].value_counts()
    tab = pd.DataFrame({'first_episodes': fm, 'all_episodes': base}).fillna(0)
    tab['first_mover_index'] = ((fm / fm.sum()) / (base / base.sum())).round(2)
    tab = tab[tab['all_episodes'] >= 30].sort_values('first_mover_index', ascending=False)
    print('First-mover index (>1 = over-represented as the FIRST observed type of a household):')
    display(tab)
    print('NOTE: "first" = first observed since 2021 (Wgs start); pre-2021 history is censored.')

    # ── G1b: same-creditor recurrence vs cross-creditor cascade ───────────────
    _dec = []
    for _h, g in _seq.groupby('Hash'):
        gs = g.sort_values('epnum')
        seen = set(); prev = None; prev_yr = None
        for _ty, _en in zip(gs['ep_type'], gs['end']):
            if prev is not None:
                _kind = ('same type again' if _ty == prev
                         else 'known other type' if _ty in seen else 'NEW type (cascade)')
                _dec.append((prev, _kind, prev_yr))
            seen.add(_ty); prev = _ty; prev_yr = _en.year if pd.notna(_en) else None
    if len(_dec) >= 100:
        _dd = pd.DataFrame(_dec, columns=['current', 'transition', 'year'])
        _ov = _dd['transition'].value_counts(normalize=True) * 100
        print('\nG1b - what kind of transition is recurrence? (all transitions):')
        print(_ov.round(1).rename('%').to_string())
        _bt = pd.crosstab(_dd['current'], _dd['transition'], normalize='index') * 100
        _cnt2 = _dd['current'].value_counts()
        _bt = _bt.loc[[i for i in _bt.index if _cnt2.get(i, 0) >= 30]]
        if len(_bt):
            print('\nPer current type (% of its transitions, n>=30):')
            display(_bt.round(1))
        _yy2 = _dd.dropna(subset=['year'])
        if len(_yy2) >= 100 and _yy2['year'].nunique() > 1:
            _ytr = pd.crosstab(_yy2['year'].astype(int), _yy2['transition'], normalize='index') * 100
            print('\nTransition mix per year (%, year of the earlier episode):')
            print(_ytr.round(1).to_string())
        print("'same type again' = unresolved issue or reporting rhythm (see H2);")
        print("'NEW type (cascade)' = distress spreading to a creditor never seen before -")
        print('the genuine early-warning evidence.')
    else:
        print('\nG1b - too few transitions for the decomposition.')


### 14b. G2 — Breadth vs Depth

Round 1 found debt *amount* barely predicts recurrence. Hypothesis: the **number of distinct
creditor types** a household has accumulated is the real severity marker. Both enter one model
head-to-head; if breadth dominates, triage should count creditors, not read balances.


In [ ]:
_report_section('14b. G2 - Breadth vs Depth')
# breadth is computed in the §13 prep cell (shared with E1b); recompute only if missing
if 'breadth_band' not in eff.columns:
    _ts2 = signalen_x.dropna(subset=['Hash', 'Datum melding', 'Type melder']).sort_values('Datum melding')
    _by_h = {h: list(zip(g['Datum melding'].tolist(), g['Type melder'].tolist()))
             for h, g in _ts2.groupby('Hash')}
    def _breadth(h, t):
        lst = _by_h.get(h)
        if not lst or pd.isna(t):
            return np.nan
        return len({ty for d0, ty in lst if d0 <= t})
    eff['breadth'] = [_breadth(h, t) for h, t in zip(eff['Hash'], eff['signal_date'])]
    eff['breadth_band'] = pd.cut(eff['breadth'], bins=[0, 1, 2, 99], labels=['1', '2', '3+'])

print('90-day return by creditor breadth at case start:')
for b, g in eff.dropna(subset=['breadth_band']).groupby('breadth_band'):
    if len(g) >= 30:
        print(f'  {b:>3} creditor type(s): {cuminc_str(g["duration"], g["event"], 90, 5)}  (n={len(g):,})')

print('\nHead-to-head Cox (breadth and debt amount in ONE model):')
if eff['breadth_band'].dropna().nunique() < 2:
    hr = None
    print('  Creditor breadth does not vary in this dataset - Cox skipped.')
else:
    hr = cox_hr(eff, 'duration ~ C(breadth_band) + C(debt_band) + recidive_hist',
                ['breadth_band', 'debt_band', 'recidive_hist'])
if hr is not None:
    display(hr)
    print('If breadth HRs are large/significant while debt-band HRs stay ~1:')
    print('count creditors, not euros.')


### 14c. G3 — Trajectory Typology

Maps the recurring population. Base classes by volume — **one-off** (1 episode), **episodic**
(2–3), **chronic** (4+) — plus four independent behavioural flags per household:
**escalating** (gaps shortening), **resolving** (gaps lengthening or inactive ≥1 year),
**monthly cadence** (administrative re-reporting pattern) and **cascading** (a later episode
introduced a new creditor type). The stakeholder headline: what share of all signal volume
comes from the most chronic 5–10% of households?


In [ ]:
_report_section('14c. G3 - Trajectory Typology')
_hh = epi.groupby('Hash').agg(n_ep=('epnum', 'max'), last=('end', 'max'),
                              n_types=('ep_type', 'nunique')).reset_index()
_hh['cls'] = np.where(_hh['n_ep'] == 1, 'one-off',
              np.where(_hh['n_ep'] <= 3, 'episodic (2-3)', 'chronic (4+)'))
_hh['inactive_365d'] = (EXPORT_DATE - _hh['last']).dt.days >= 365

# gap-trend flag (same rule as E2)
_g = epi.dropna(subset=['gap_days'])
_trend = {}
for _h, gg in _g.groupby('Hash'):
    v = gg.sort_values('epnum')['gap_days'].values
    if len(v) >= 2:
        _trend[_h] = ('escalating' if v[-1] < v[0] * 0.75
                      else 'resolving' if v[-1] > v[0] * 1.33 else 'stable')
_hh['gap_trend'] = _hh['Hash'].map(_trend)

# monthly-cadence flag (same rule as 14e): >=50% of signal gaps 25-35d, <=2 creditor types
_sg = signalen_x.dropna(subset=['Hash', 'Datum melding']).sort_values(['Hash', 'Datum melding']).copy()
_sg['_gap'] = _sg.groupby('Hash')['Datum melding'].diff().dt.days
_cad = _sg.dropna(subset=['_gap']).groupby('Hash')['_gap'].apply(lambda s: (s.between(25, 35)).mean() >= 0.5)
_nty = signalen_x.dropna(subset=['Hash']).groupby('Hash')['Type melder'].nunique()
_hh['flag_monthly_cadence'] = _hh['Hash'].map(_cad).fillna(False) & (_hh['Hash'].map(_nty) <= 2)
_hh['flag_escalating'] = _hh['gap_trend'] == 'escalating'
_hh['flag_resolving']  = (_hh['gap_trend'] == 'resolving') | ((_hh['n_ep'] >= 2) & _hh['inactive_365d'])
_hh['flag_cascading']  = _hh['n_types'] >= 2   # a later episode introduced a NEW creditor type

_prof = signalen_x.dropna(subset=['Hash']).groupby('Hash').agg(
    n_sig=('SignaalID', 'count'),
    children=('Minderjarige kinderen', lambda s: (s.astype(str).str.lower() == 'ja').any()))
_hh = _hh.merge(_prof, on='Hash', how='left')

print('Trajectory classes x behavioural flags (flags are independent, not exclusive):')
rows = []
for c, g in _hh.groupby('cls'):
    rows.append((c, len(g), round(len(g)/len(_hh)*100, 1), round(g['n_sig'].mean(), 1),
                 round(g['children'].mean()*100, 1),
                 round(g['flag_escalating'].mean()*100, 1),
                 round(g['flag_resolving'].mean()*100, 1),
                 round(g['flag_monthly_cadence'].mean()*100, 1),
                 round(g['flag_cascading'].mean()*100, 1)))
tab = pd.DataFrame(rows, columns=['class', 'households', '% of hh', 'mean signals', '% children',
                                  '% escalating', '% resolving', '% monthly cadence', '% cascading'])
display(tab)
print('escalating = inter-episode gaps shortening | resolving = gaps lengthening or inactive >=1y')
print('monthly cadence = >=50% of gaps 25-35d from <=2 creditors (administrative re-reporting)')
print('cascading = a later episode introduced a new creditor type (multi-creditor spread)')

fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(tab['class'], tab['% of hh'], color=COLORS[0])
for _b, v in zip(ax.patches, tab['% of hh']):
    ax.text(_b.get_width() + 0.5, _b.get_y() + _b.get_height()/2, f'{v}%', va='center', fontsize=9)
ax.set_xlabel('% of households'); ax.set_title('G3 - Trajectory classes')
plt.tight_layout(); plt.show()

_sc = _prof['n_sig'].sort_values(ascending=False)
_tot = _sc.sum()
for pct in (5, 10):
    k = max(int(len(_sc) * pct / 100), 1)
    print(f"Top {pct:>2}% households ({k:,}) generate {_sc.head(k).sum()/_tot*100:.1f}% of all signals")

_first_ep_yr = epi.groupby('Hash')['start'].min().dt.year
_hh['_cohort'] = _hh['Hash'].map(_first_ep_yr)
_cc = _hh.dropna(subset=['_cohort']).copy()
if _cc['_cohort'].nunique() > 1:
    _ct = pd.crosstab(_cc['_cohort'].astype(int), _cc['cls'], normalize='index') * 100
    print('\nClass shares by first-episode year (cohort view, %):')
    print(_ct.round(1).to_string())
    print('(later cohorts have had less time to accumulate episodes - compare with care)')


### 14d. G5 — Early-Warning Value per Signal Partner

One scorecard per partner type: how often it opens a household's history, how often another
episode follows, how fast, how reachable its cases are, and how often they end in real
resolution. Caveat: municipal handling differs per partner (forwarding 76–99%), so reach and
resolution mix partner signal quality with municipal triage.


In [ ]:
_report_section('14d. G5 - Early-Warning Value per Signal Partner')
_first_ty = epi.dropna(subset=['ep_type']).sort_values(['Hash', 'epnum']).groupby('Hash')['ep_type'].first()
_fm_share = _first_ty.value_counts(normalize=True) * 100
rows = []
for _ty in epi['ep_type'].dropna().unique():
    ep_t = epi[epi['ep_type'] == _ty]
    if len(ep_t) < 30:
        continue
    followed = ep_t['gap_event'].mean() * 100
    med_gap  = ep_t.loc[ep_t['gap_event'] == 1, 'gap_days'].median()
    ge = eff[eff['Type melder'] == _ty]
    reach = ge['reached'].mean() * 100 if len(ge) >= 30 else np.nan
    if len(ge) >= 30 and 'eindresultaat' in ge.columns:
        _res  = ge['eindresultaat'].astype(str).str.lower()
        resol = (_res.str.contains('betaald') | _res.str.contains('betalingsregeling')).mean() * 100
    else:
        resol = np.nan
    rows.append((str(_ty)[:26], len(ep_t), round(_fm_share.get(_ty, 0), 1), round(followed, 1),
                 med_gap, round(reach, 1) if pd.notna(reach) else np.nan,
                 round(resol, 1) if pd.notna(resol) else np.nan))
if rows:
    tab = pd.DataFrame(rows, columns=['partner type', 'episodes', '% households: first type',
                                      '% followed by next episode', 'median days to next',
                                      'reach % (its cases)', '% paid/arrangement'])
    display(tab.sort_values('% households: first type', ascending=False))
    print('High "first type" + high "followed by next" = genuinely early warning signal.')
    print('Caveat: reach/resolution reflect municipal triage per partner as well as the partner itself.')
else:
    print('Too few episodes per partner type.')


### 14e. Deep-Dive: The 10 Most-Signalling Households

High-volume households are a normal part of the data — none are excluded. This cell profiles
the ten most-signalling households to show what kind of recurrence they represent. The
timeline plot separates two patterns: evenly spaced ~30-day dots from one or two creditors =
**administrative re-reporting**; clustered bursts across several creditors = **genuine
repeated distress**. Households are labelled by rank only — hashes are never shown.


In [ ]:
_report_section('14e. Deep-Dive: The 10 Most-Signalling Households')
_cnt = signalen_x.dropna(subset=['Hash']).groupby('Hash').size().sort_values(ascending=False)
TOP10 = _cnt.head(10)
print(f"Top 10 households: {TOP10.sum():,} of {int(_cnt.sum()):,} signals "
      f"({TOP10.sum()/_cnt.sum()*100:.1f}% of total volume)")

rows = []
for _rank, (_h, _n) in enumerate(TOP10.items(), 1):
    g = signalen_x[signalen_x['Hash'] == _h].sort_values('Datum melding')
    dts = g['Datum melding'].dropna()
    span_m = max((dts.max() - dts.min()).days / 30.44, 1) if len(dts) > 1 else 1
    gaps = dts.diff().dt.days.dropna()
    monthly_pct = (gaps.between(25, 35)).mean() * 100 if len(gaps) else np.nan
    ntypes = int(g['Type melder'].nunique())
    _ep = epi[epi['Hash'] == _h]
    n_ep = int(_ep['epnum'].max()) if len(_ep) else 0
    ge = eff[eff['Hash'] == _h]
    if pd.notna(monthly_pct) and monthly_pct >= 50 and ntypes <= 2:
        pattern = 'monthly re-reporting'
    elif n_ep >= 3:
        pattern = 'episodic bursts'
    else:
        pattern = 'mixed / single episode'
    rows.append((f'HH-{_rank}', _n, round(span_m), round(_n / span_m, 1), ntypes, n_ep,
                 round(gaps.median()) if len(gaps) else np.nan,
                 round(monthly_pct) if pd.notna(monthly_pct) else np.nan,
                 'yes' if (len(ge) and ge['reached'].any()) else 'no',
                 'yes' if (len(ge) and (ge['arm'] == 'C_accepted').any()) else 'no',
                 int((EXPORT_DATE - dts.max()).days) if len(dts) else np.nan,
                 pattern))
tab = pd.DataFrame(rows, columns=['household', 'signals', 'span (months)', 'signals/month',
                                  'creditor types', 'episodes', 'median gap (d)', '% gaps 25-35d',
                                  'ever reached', 'ever accepted help', 'days since last signal',
                                  'pattern'])
display(tab)
print("'monthly re-reporting' = regular ~30d gaps from <=2 creditors -> administrative cadence,")
print("likely one unresolved debt re-flagged month after month.")
print("'episodic bursts' = distinct episodes -> genuine repeated distress.")

# ── Signal timelines ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 5))
for i, (_h, _n) in enumerate(TOP10.items()):
    dts = signalen_x.loc[signalen_x['Hash'] == _h, 'Datum melding'].dropna()
    ax.scatter(dts, [len(TOP10) - i] * len(dts), s=14, color=COLORS[i % len(COLORS)])
ax.set_yticks(range(len(TOP10), 0, -1))
ax.set_yticklabels([f'HH-{r}' for r in range(1, len(TOP10) + 1)])
ax.set_xlabel('date'); ax.set_title('Signal timelines - 10 most-signalling households')
plt.tight_layout(); plt.show()

# ── Are heavy households treated / responding differently? ────────────────────
_heavy = set(TOP10.index)
gh = eff[eff['Hash'].isin(_heavy)]; gr = eff[~eff['Hash'].isin(_heavy)]
if len(gh) >= 10 and len(gr) >= 100:
    print('\nHeavy-tail vs all other households (closed cases):')
    print(f"  cases:            {len(gh):>6,}  vs {len(gr):>6,}")
    print(f"  reach rate:       {gh['reached'].mean()*100:5.1f}%  vs {gr['reached'].mean()*100:5.1f}%")
    print(f"  help accepted:    {(gh['arm']=='C_accepted').mean()*100:5.1f}%  vs {(gr['arm']=='C_accepted').mean()*100:5.1f}%")
    print(f"  90d return:       {cuminc_str(gh['duration'], gh['event'], 90, 5)}  vs {cuminc_str(gr['duration'], gr['event'], 90, 5)}")
    print('If heavy households are rarely reached, the system may be repeatedly signalling')
    print('exactly the households it never manages to contact.')
else:
    print('\nToo few closed cases among the top-10 households for a comparison.')


## 15. Module 3 — Policy Shocks & Operational Planning (H2, F1, F2, H1)

H2 comes first: it measures how much of "recurrence" is reporting rhythm, which calibrates the
headline metric and is a prerequisite for reading F1's trends.


### 15a. H2 — Reporting Cadence per Partner (metric calibration)

How much of measured recurrence is household behaviour, and how much is partners' reporting
rhythm? Measures inter-report intervals per partner and recomputes the headline return rate
under alternative minimum-gap definitions — the spread is the administrative share.


In [ ]:
_report_section('15a. H2 - Reporting Cadence per Partner')
_ts3 = (signalen_x.dropna(subset=['Hash', 'Datum melding', 'Type melder'])
        .sort_values(['Hash', 'Type melder', 'Datum melding']).copy())
_ts3['_gap'] = _ts3.groupby(['Hash', 'Type melder'])['Datum melding'].diff().dt.days
gaps = _ts3.dropna(subset=['_gap'])
gaps = gaps[gaps['_gap'] > 0]
rows = []
for _ty, g in gaps.groupby('Type melder'):
    if len(g) < 50:
        continue
    rows.append((str(_ty)[:26], len(g), g['_gap'].median(),
                 round((g['_gap'].between(25, 35)).mean() * 100, 1),
                 round((g['_gap'] <= MIN_GAP_DAYS).mean() * 100, 1)))
if rows:
    print('Inter-report intervals within the same household & partner type:')
    display(pd.DataFrame(rows, columns=['partner type', 'intervals', 'median days',
                                        '% in 25-35d (monthly cadence)', f'% <= {MIN_GAP_DAYS}d']))
else:
    print('Too few repeated same-type reports.')

if len(gaps) >= 400 and gaps['Datum melding'].dt.year.nunique() > 1:
    gaps['_yr'] = gaps['Datum melding'].dt.year
    _keepc = [t for t in gaps['Type melder'].unique() if (gaps['Type melder'] == t).sum() >= 200]
    if _keepc:
        _pvh = (gaps[gaps['Type melder'].isin(_keepc)]
                .pivot_table(index='Type melder', columns='_yr', values='_gap', aggfunc='median'))
        print('\nMedian inter-report interval per partner per year (days) - cadence stability:')
        print(_pvh.round(0).to_string())

print('\nSensitivity: 90-day case return rate under alternative minimum-gap definitions:')
for gp in [7, 14, 30, 60]:
    _nx = []
    for _h, _c in zip(eff['Hash'], eff['close_date']):
        arr = _sig_by_hash.get(_h)
        if not arr:
            _nx.append(pd.NaT); continue
        i = bisect.bisect_right(arr, _c + pd.Timedelta(days=gp))
        _nx.append(arr[i] if i < len(arr) else pd.NaT)
    _nx = pd.Series(_nx, index=eff.index)
    _ev = _nx.notna().astype(int)
    _du = (_nx.fillna(EXPORT_DATE) - eff['close_date']).dt.days
    _ok = _du > 0
    print(f'  gap >{gp:>2}d: 90d return = {cuminc_str(_du[_ok], _ev[_ok], 90, 5)}   '
          f'(events {int(_ev[_ok].sum()):,})')
print('The spread across gap definitions bounds the share of "recurrence" that is reporting rhythm.')


### 15b. F1 — Energy Crisis 2022: Comparative Trends

Did the energy price shock create *new* distressed households or intensify existing ones, and
did relief (energietoeslag late 2022, price cap Jan 2023) show up in the series? Water,
health-insurer and housing signals serve as comparison series. Framed as comparative trends —
**not** clean difference-in-differences. The 2021 ramp (Wgs partner onboarding) is excluded.


In [ ]:
_report_section('15b. F1 - Energy Crisis 2022: Comparative Trends')
_s = signalen_x.dropna(subset=['Datum melding', 'Type melder']).copy()
_s = _s[_s['Datum melding'] >= pd.Timestamp('2021-07-01')]     # skip Wgs onboarding ramp
if len(_s) < 500 or _s['Datum melding'].dt.year.nunique() < 2:
    print('Data span too short for the crisis analysis - skipped.')
else:
    _s['month'] = _s['Datum melding'].dt.to_period('M').dt.to_timestamp()
    _piv = _s.pivot_table(index='month', columns='Type melder', values='SignaalID',
                          aggfunc='count').fillna(0)
    _keep = [c for c in _piv.columns if _piv[c].sum() >= 200]
    fig, ax = plt.subplots(figsize=(13, 5))
    for i, c in enumerate(_keep):
        ax.plot(_piv.index, _piv[c].rolling(3, center=True).mean(),
                label=str(c)[:26], linewidth=2, color=COLORS[i % len(COLORS)])
    for d0, lab in [('2022-03-01', 'price surge'), ('2022-11-01', 'energietoeslag'),
                    ('2023-01-01', 'price cap')]:
        if pd.Timestamp(d0) > _piv.index.min():
            ax.axvline(pd.Timestamp(d0), color='red', linestyle='--', alpha=0.5)
            ax.text(pd.Timestamp(d0), ax.get_ylim()[1] * 0.97, lab, rotation=90,
                    fontsize=8, va='top')
    ax.set_title('F1 - Monthly signals per partner type (3-month rolling mean)')
    ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()

    _per = [('2021H2', '2021-07-01', '2022-01-01'), ('2022 crisis', '2022-01-01', '2023-01-01'),
            ('2023 cap', '2023-01-01', '2024-01-01'), ('2024+', '2024-01-01', '2027-01-01')]
    rows = []
    for lab, a, b in _per:
        m = _s[(_s['Datum melding'] >= a) & (_s['Datum melding'] < b)]
        if len(m) == 0:
            continue
        months = max((pd.Timestamp(b) - pd.Timestamp(a)).days / 30.44, 1)
        months = min(months, max((m['Datum melding'].max() - pd.Timestamp(a)).days / 30.44, 1))
        r0 = {'period': lab}
        for c in _keep:
            r0[str(c)[:20]] = round((m['Type melder'] == c).sum() / months, 1)
        rows.append(r0)
    print('Signals per month by period (values behind the chart):')
    display(pd.DataFrame(rows).set_index('period'))

    _pm = _piv[_keep].astype(int).copy()
    _pm.index = _pm.index.strftime('%Y-%m')
    print('Monthly signal counts per partner type (full series behind the chart,')
    print('for future re-analysis of the trends):')
    print(_pm.to_string())

    _first_seen = signalen_x.dropna(subset=['Hash', 'Datum melding']).groupby('Hash')['Datum melding'].min()
    _s['is_new'] = _s['Datum melding'] <= _s['Hash'].map(_first_seen)
    _is_en = _s['Type melder'].astype(str).str.lower().str.contains('energie')
    rows = []
    for lab, a, b in _per:
        m = _s[(_s['Datum melding'] >= a) & (_s['Datum melding'] < b)]
        en = m[_is_en.reindex(m.index, fill_value=False)]
        ot = m[~_is_en.reindex(m.index, fill_value=False)]
        if len(en) >= 30 and len(ot) >= 30:
            rows.append((lab, round(en['is_new'].mean() * 100, 1), round(ot['is_new'].mean() * 100, 1)))
    if rows:
        print('\nShare of signals from FIRST-TIME households (new inflow vs known population):')
        display(pd.DataFrame(rows, columns=['period', 'energy signals: % new hh', 'other signals: % new hh']))
    print('Caveats: comparative trends, not DiD; "new" = first observed since 2021;')
    print('check H2 that partner reporting practices were stable before reading effects.')


### 15c. F2 — Seasonality by Mechanism

Do signal types follow their expected seasonal mechanisms (energy after winter bills, health
insurance after the January premium reset, water as the low-seasonality control)? Directly
usable for outreach capacity planning. The 2021 ramp year is excluded.


In [ ]:
_report_section('15c. F2 - Seasonality by Mechanism')
_s2 = signalen_x.dropna(subset=['Datum melding', 'Type melder']).copy()
_s2 = _s2[_s2['Datum melding'] >= pd.Timestamp('2022-01-01')]
if len(_s2) < 500 or _s2['Datum melding'].dt.year.nunique() < 2:
    print('Data span too short for a seasonality profile - skipped.')
else:
    _s2['cal_month'] = _s2['Datum melding'].dt.month
    _pv = _s2.pivot_table(index='cal_month', columns='Type melder', values='SignaalID',
                          aggfunc='count').fillna(0)
    _keep = [c for c in _pv.columns if _pv[c].sum() >= 300]
    if not _keep:
        print('No partner type with enough volume.')
    else:
        _idx = (_pv[_keep] / _pv[_keep].mean()) * 100
        fig, ax = plt.subplots(figsize=(12, 5))
        for i, c in enumerate(_keep):
            ax.plot(_idx.index, _idx[c], marker='o', label=str(c)[:24], color=COLORS[i % len(COLORS)])
        ax.axhline(100, color='gray', linestyle='--')
        ax.set_xticks(range(1, 13)); ax.set_xlabel('calendar month')
        ax.set_title('F2 - Seasonality index per partner type (100 = type average, 2022+)')
        ax.legend(fontsize=8)
        plt.tight_layout(); plt.show()
        print('Seasonality index (values behind the chart):')
        display(_idx.round(0))
    eff['_close_q'] = eff['close_date'].dt.quarter
    print('\n90-day return by case-close quarter:')
    for q, g in eff.dropna(subset=['_close_q']).groupby('_close_q'):
        if len(g) >= 100:
            print(f'  Q{int(q)}: {cuminc_str(g["duration"], g["event"], 90, 5)}  (n={len(g):,})')


### 15d. H1 — Risk Stratification at First Contact

Which first-episode features are associated with becoming a chronic returner (≥3 episodes
within 18 months)? Interpretable logistic model on intake-time features only, with **temporal
validation** (train on households first seen before 2024, test on 2024+) and calibration —
not just AUC. Framing: transparent risk-stratification research, not a decision model.

**H1b** adds a second target: the top 10% of households by signal volume within 18 months
(threshold from training years only) — predicting future *workload*, which partly reflects
reporting cadence (see H2).


In [ ]:
_report_section('15d. H1 - Risk Stratification at First Contact')
_FOLLOW = 548   # 18 months
_h1 = epi.groupby('Hash').agg(first=('start', 'min')).reset_index()
_h1 = _h1[_h1['first'] <= EXPORT_DATE - pd.Timedelta(days=_FOLLOW)]
_epw = epi.merge(_h1, on='Hash')
_epw = _epw[_epw['start'] <= _epw['first'] + pd.Timedelta(days=_FOLLOW)]
_h1['y'] = (_h1['Hash'].map(_epw.groupby('Hash')['epnum'].max()) >= 3).astype(int)

# H1b target: signal volume within the same 18-month window (workload burden)
_sw = signalen_x.dropna(subset=['Hash', 'Datum melding']).merge(_h1[['Hash', 'first']], on='Hash')
_sw = _sw[(_sw['Datum melding'] >= _sw['first']) &
          (_sw['Datum melding'] <= _sw['first'] + pd.Timedelta(days=_FOLLOW))]
_h1['sig18'] = _h1['Hash'].map(_sw.groupby('Hash').size()).fillna(0)

_fs = (signalen_x.dropna(subset=['Hash', 'Datum melding']).sort_values('Datum melding')
       .groupby('Hash').first().reset_index())
_h1 = _h1.merge(_fs[['Hash', 'Type melder', 'age_band', 'debt_band', 'Minderjarige kinderen']],
                on='Hash', how='left')
_h1['children'] = ja_nee(_h1['Minderjarige kinderen']).astype(float)
d = _h1.dropna(subset=['Type melder', 'age_band', 'debt_band', 'children']).copy()

train = d[d['first'] <  pd.Timestamp('2024-01-01')].copy()
test  = d[d['first'] >= pd.Timestamp('2024-01-01')].copy()
print('Inclusion flow (who this analysis describes):')
print(f'  households with >=1 episode:              {epi["Hash"].nunique():,}')
print(f'  first episode >= {_FOLLOW}d before export:  {len(_h1):,}')
print(f'  with complete first-episode features:     {len(d):,}')
if len(d):
    print(f'  chronic (>=3 episodes in 18mo):           {d["y"].mean()*100:.1f}%')
print(f'  temporal split: train (<2024) n={len(train):,} | test (2024+) n={len(test):,}')
if len(d) and d['first'].dt.year.nunique() > 1:
    _cy1 = (d.assign(_yr=d['first'].dt.year).groupby('_yr')['y']
            .agg(households='count', chron='mean'))
    _cy1['chronic_%'] = (_cy1['chron'] * 100).round(1)
    print('\nChronic rate by first-episode cohort year (each cohort has the same 18-month window):')
    print(_cy1.drop(columns='chron').to_string())

def _auc(y, p):
    y = np.asarray(y); p = np.asarray(p)
    r = pd.Series(p).rank().values
    n1 = y.sum(); n0 = len(y) - n1
    return (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0) if n1 * n0 > 0 else np.nan

if len(train) >= 300 and train['y'].sum() >= 30 and len(test) >= 100 and test['y'].nunique() > 1:
    import statsmodels.formula.api as smf
    try:
        m = smf.logit('y ~ C(Q("Type melder")) + C(age_band) + C(debt_band) + children',
                      data=train).fit(disp=0)
        pt = m.predict(train); pv = m.predict(test)
        print(f"\nAUC train = {_auc(train['y'], pt):.3f}   AUC test (2024+) = {_auc(test['y'], pv):.3f}")
        _cb = pd.DataFrame({'p': pv, 'y': test['y'].values})
        _cb['bin'] = pd.qcut(_cb['p'], 5, duplicates='drop')
        cal = _cb.groupby('bin', observed=True).agg(predicted=('p', 'mean'),
                                                    observed=('y', 'mean'), n=('y', 'size'))
        print('\nCalibration on the test set (chronic rate, %):')
        display((cal[['predicted', 'observed']] * 100).round(1).join(cal['n']))
        _or = pd.DataFrame({'OR': np.exp(m.params), 'p': m.pvalues.round(3)}).drop(
            index='Intercept', errors='ignore').round(3)
        print('Feature associations (odds ratios, train set):')
        display(_or.sort_values('OR', ascending=False).head(12))

        # ── H1b: heavy-burden target (top 10% by signals in first 18 months) ──
        _thr = train['sig18'].quantile(0.90)
        train['y2'] = (train['sig18'] >= _thr).astype(int)
        test['y2']  = (test['sig18']  >= _thr).astype(int)
        print(f"\nH1b - heavy-burden target: top 10% by signal volume within {_FOLLOW}d")
        print(f"      (threshold >= {_thr:.0f} signals, computed on TRAIN years only - no leakage)")
        if train['y2'].sum() >= 20 and test['y2'].nunique() > 1:
            try:
                m2 = smf.logit('y2 ~ C(Q("Type melder")) + C(age_band) + C(debt_band) + children',
                               data=train).fit(disp=0)
                print(f"AUC train = {_auc(train['y2'], m2.predict(train)):.3f}   "
                      f"AUC test (2024+) = {_auc(test['y2'], m2.predict(test)):.3f}")
                _or2 = pd.DataFrame({'OR': np.exp(m2.params), 'p': m2.pvalues.round(3)}).drop(
                    index='Intercept', errors='ignore').round(3)
                display(_or2.sort_values('OR', ascending=False).head(10))
            except Exception as _e2:
                print(f'  H1b model failed: {_e2}')
        else:
            print('  Too few heavy-burden households in train, or no variation in test - skipped.')
        print('Caveat: signal volume partly reflects creditor reporting cadence (see H2), so')
        print('H1b predicts future WORKLOAD - not purely household behaviour.')
    except Exception as _e:
        print(f'Model failed: {_e}')
else:
    print('Insufficient data for temporal validation - model skipped.')
print('\nFraming: transparent risk stratification for prioritisation research - NOT a decision model.')


### 15e. W — Workflow: When Are Households Receptive, and Which Method When?

**Hypothesis:** households contacted at the very start of a problem often refuse help ("I can
fix this myself"); after months of persistent signals (recidive maand-2+ positive) they are
more receptive. If true, municipalities can allocate cheap methods to the early phase and
reserve expensive ones (phone, home visit) for the receptive phase.
**W1** measures help acceptance among *reached* cases by problem persistence (case number,
maand-depth, tenure, contact speed), with an adjusted model and a within-household check.
**W2** builds the workflow matrix: *accepted help per 100 attempts* per method × phase —
reach probability × acceptance combined, since a method that never reaches cannot harvest
late receptiveness. NB: the Wgs 4-week duty means this informs **method choice**, not delay.


In [ ]:
_report_section('15e. W - Workflow: Receptiveness Timing & Method Allocation')
# W1: is help acceptance higher when the problem is older/more persistent?
_rc = eff_sorted[eff_sorted['reached'] == True].copy()
_rc['accepted'] = (_rc['arm'] == 'C_accepted')
print(f"Population: {len(_rc):,} reached closed cases "
      f"({int(_rc['accepted'].sum()):,} accepted help = {_rc['accepted'].mean()*100:.1f}%)")
print("NB: the Wgs requires municipal action within 4 weeks of each signal - this experiment")
print("informs the CHOICE OF METHOD per phase, never delaying contact.\n")

# persistence measures at the case's opening signal
_mcols2 = [c for c in signalen.columns if c.startswith('Recidive maand')]
if _mcols2:
    _sl2 = signalen.dropna(subset=['Meldingnummer']).sort_values('Datum melding')
    _dep = _sl2.groupby('Meldingnummer')[_mcols2].first()
    _depth_n = sum((_dep[c].astype(str).str.strip().str.lower() == 'ja') for c in _mcols2)
    _rc = _rc.merge(_depth_n.rename('maand_depth'), left_on='Meldingnummer',
                    right_index=True, how='left')
    _rc['depth_band'] = pd.cut(_rc['maand_depth'], bins=[-1, 0, 1, 99],
                               labels=['0 (new problem)', '1 month', '2+ (persistent)'])
_first_seen2 = signalen.dropna(subset=['Hash', 'Datum melding']).groupby('Hash')['Datum melding'].min()
_rc['tenure_d'] = (_rc['signal_date'] - _rc['Hash'].map(_first_seen2)).dt.days
_rc['tenure_band'] = pd.cut(_rc['tenure_d'], bins=[-1, 0, 60, 180, 1e9],
                            labels=['first signal', '1-60d', '61-180d', '>180d'])
_rc['epn_band'] = pd.cut(_rc['case_epnum'], bins=[0, 1, 2, 99], labels=['1st case', '2nd', '3rd+'])
_rc['reachdel_band'] = pd.cut(_rc['days_to_reach'], bins=[-1, 14, 30, 1e9],
                              labels=['<=14d', '15-30d', '>30d'])

print('W1 - Help acceptance among reached cases, by problem persistence (n>=30 per band):')
for _col, _lab in [('epn_band', 'Case number for this household'),
                   ('depth_band', 'Recidive-maand depth of the opening signal'),
                   ('tenure_band', 'Time since household first observed'),
                   ('reachdel_band', 'Days from signal to first successful contact')]:
    if _col not in _rc.columns:
        continue
    g = _rc.dropna(subset=[_col]).groupby(_col, observed=True)['accepted'].agg(n='count', acc='mean')
    g = g[g['n'] >= 30]
    if len(g) >= 2:
        print(f'\n- {_lab}:')
        print(g.assign(**{'accepted_%': (g['acc'] * 100).round(1)}).drop(columns='acc').to_string())

_rc['_yr'] = _rc['close_date'].dt.year
if _rc['_yr'].nunique() > 1:
    _ya = _rc.groupby('_yr')['accepted'].agg(n='count', acc='mean')
    _ya = _ya[_ya['n'] >= 30]
    if len(_ya) > 1:
        print('\n- Acceptance among reached cases per close year:')
        print(_ya.assign(**{'accepted_%': (_ya['acc'] * 100).round(1)}).drop(columns='acc').to_string())

# adjusted check: persistence effect net of severity markers
import statsmodels.formula.api as smf
if 'depth_band' in _rc.columns:
    _d = _rc.dropna(subset=['depth_band', 'debt_band', 'age_band']).copy()
    if len(_d) >= 300 and _d['accepted'].sum() >= 30 and _d['depth_band'].nunique() > 1:
        try:
            _d['acc_f'] = _d['accepted'].astype(float)
            _d['depth_band'] = _d['depth_band'].cat.remove_unused_categories()
            m = smf.logit('acc_f ~ C(depth_band) + C(debt_band) + C(age_band) + recidive_hist',
                          data=_d).fit(disp=0)
            _or = pd.DataFrame({'OR': np.exp(m.params), 'p': m.pvalues.round(3)}).drop(
                index='Intercept', errors='ignore').round(3)
            print('\nAdjusted odds ratios for acceptance (ref = new problem; controls: debt, age,')
            print('prior recidive). OR>1 for persistent = receptiveness grows with problem age:')
            display(_or)
        except Exception as _e:
            print(f'\nAdjusted model failed: {_e}')
    else:
        print('\nAdjusted model skipped - too few complete reached cases.')

# within-household: among households reached in multiple episodes with DIFFERENT responses,
# did acceptance come in the LATER episode?
_wd = []
for _h, g in _rc.groupby('Hash'):
    if g['accepted'].nunique() > 1:
        _wd.append(g.loc[g['accepted'], 'case_epnum'].min() > g.loc[~g['accepted'], 'case_epnum'].min())
if len(_wd) >= 20:
    print(f"\nWithin-household check ({len(_wd)} households reached in multiple episodes with")
    print(f"different responses): acceptance came in the LATER episode for {np.mean(_wd)*100:.0f}%.")
    print("(>50% supports 'receptiveness grows with persistence' within the same household)")
else:
    print(f"\nWithin-household check: only {len(_wd)} discordant households - too few to read.")

# ── W2: method x phase workflow matrix ────────────────────────────────────────
print('\nW2 - Which method pays off in which phase? (accepted help per 100 attempts)')
if len(contacts):
    _phase_map = dict(zip(eff_sorted['Meldingnummer'],
                          np.where(eff_sorted['case_epnum'] >= 2, 'repeat episode', 'first episode')))
    _ca = contacts.dropna(subset=['Meldingnummer', 'Wijze van contactpoging']).copy()
    _ca['_phase'] = _ca['Meldingnummer'].map(_phase_map)
    _ca = _ca.dropna(subset=['_phase'])
    _ca['_r'] = _ca['_reached'] == True
    _ac = _rc.dropna(subset=['reach_method']).copy()
    _ac['_phase'] = np.where(_ac['case_epnum'] >= 2, 'repeat episode', 'first episode')
    rows = []
    for _ph in ['first episode', 'repeat episode']:
        _cp = _ca[_ca['_phase'] == _ph]; _ap = _ac[_ac['_phase'] == _ph]
        for _m in _cp['Wijze van contactpoging'].value_counts().index:
            _att = _cp[_cp['Wijze van contactpoging'] == _m]
            _re  = _ap[_ap['reach_method'] == _m]
            if len(_att) < 100 or len(_re) < 20:
                continue
            reach = _att['_r'].mean(); acc = _re['accepted'].mean()
            rows.append((_ph, str(_m)[:38], len(_att), round(reach * 100, 1), len(_re),
                         round(acc * 100, 1), round(reach * acc * 100, 2)))
    if rows:
        _wt = pd.DataFrame(rows, columns=['phase', 'method', 'attempts', 'reach %',
                                          'reached cases', 'acceptance % of reached',
                                          'accepted per 100 attempts'])
        display(_wt.sort_values(['phase', 'accepted per 100 attempts'], ascending=[True, False]))
        print('Best value per phase (accepted help per 100 attempts):')
        for _ph, g in _wt.groupby('phase'):
            _b = g.sort_values('accepted per 100 attempts', ascending=False).iloc[0]
            print(f"  {_ph:<15} -> {_b['method']}  ({_b['accepted per 100 attempts']} per 100 attempts)")
        print('\nCaveats: closed cases only; observational - persistent households may be needier,')
        print('not merely readier; reach probability dominates the value column, so a cheap method')
        print('that rarely reaches cannot harvest late receptiveness. Weigh against staff cost per')
        print('attempt (letter/email << phone << home visit) before changing the workflow.')
    else:
        print('  Too few attempts / reached cases per method-phase cell.')
else:
    print('  No contact-attempt data available.')


## 16. Generate the Results File

Run this cell **last**. It writes one self-contained, timestamped HTML file containing every
table, chart and printed value from above — this is the only file to send back. It opens in any
browser (Print → Save as PDF if needed) and contains aggregates only, no individual records.


In [ ]:
_report_section('16. Generate the Results File to Send Back')
_doc = f'''<!DOCTYPE html>
<html><head><meta charset="utf-8"><title>ELSA Vroegsignalering — Results</title>
<style>
 body{{font-family:-apple-system,Segoe UI,Arial,sans-serif;max-width:1000px;margin:24px auto;
       padding:0 18px;color:#222;line-height:1.4}}
 h1{{border-bottom:3px solid #2c7fb8;padding-bottom:6px}}
 h2.sec{{background:#2c7fb8;color:#fff;padding:6px 12px;border-radius:4px;margin-top:34px}}
 pre{{background:#f6f8fa;padding:8px 10px;border-radius:4px;white-space:pre-wrap;
      font-size:13px;font-family:SFMono-Regular,Consolas,monospace}}
 table{{border-collapse:collapse;margin:10px 0;font-size:13px}}
 th,td{{border:1px solid #ccc;padding:4px 9px;text-align:right}}
 th{{background:#eef3f7}}
 img{{max-width:100%;margin:12px 0;border:1px solid #eee;border-radius:4px}}
 .meta{{color:#666;font-size:13px}}
</style></head><body>
<h1>ELSA Vroegsignalering — Effectiveness Results</h1>
<p class="meta">Generated {_dt.datetime.now():%Y-%m-%d %H:%M} &middot; source file: {_html.escape(str(FILE))}
&middot; {len(_REPORT_BLOCKS)} content blocks</p>
<p class="meta"><em>Aggregated statistics, tables and charts only — no individual records.</em></p>
{''.join(_REPORT_BLOCKS)}
</body></html>'''

# Write next to the notebook's working directory, timestamped so runs never overwrite
REPORT_PATH = Path(REPORT_PATH).resolve()
REPORT_PATH = REPORT_PATH.with_name(f"{REPORT_PATH.stem}_{_dt.datetime.now():%Y%m%d-%H%M%S}{REPORT_PATH.suffix}")
REPORT_PATH.write_text(_doc, encoding='utf-8')

_orig_print("=" * 70)
_orig_print("RESULTS FILE CREATED — this is the file to send back:")
_orig_print(f"   {REPORT_PATH}")
_orig_print("=" * 70)
_orig_print(f"  {len(_REPORT_BLOCKS)} content blocks captured (text, tables, charts).")
_orig_print(f"  Working directory: {Path.cwd()}")
_orig_print("  Open it in any browser; use Print -> Save as PDF if a PDF is preferred.")

# Clickable download link in Jupyter / JupyterLab
try:
    from IPython.display import FileLink
    _ipy_display(FileLink(str(REPORT_PATH)))
except Exception:
    pass

# In Google Colab, trigger a direct browser download automatically
try:
    from google.colab import files as _colab_files   # noqa
    _colab_files.download(str(REPORT_PATH))
except Exception:
    pass
